# AKANE E6：Gemmaの自信ある間違いを、辞書の説明文で直せるか

このノートは **答えを取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e6/PLAN_E6.md。

- 700個の「〜は〜の一種か」に、辞書の説明文（WordNet／Wiktionary）を足して、Gemma-2 2Bにもう一度聞き、NLI（DeBERTa大）でも判定します。
- 正解はこのノートに入っていません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**（なければ T4）。
2. 鍵マークの `HF_TOKEN` にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E6_results/E6_results.json` ができます。10〜20分の見込み。

In [ ]:
import os, json, time, urllib.parse, requests
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E6_results"
os.makedirs(OUT, exist_ok=True)
GEMMA_ID = "google/gemma-2-2b-it"
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E2-12・E3と同じ

In [ ]:
# 中身（sims/e6/e6_colab.py と同じ）
"""E6 Colabの中身：辞書の説明文を集め、Gemmaにもう一度聞き、NLIで判定する。
items: e6_items.json（700個、正解は入れない）。結果: {"meta":..., "items":[{id, wk_defs, gemma{none,S1,S2,S3}, nli{S1,S2}}]}
nli は意味ごとの [含む, 中立, 矛盾] の確率（S3＝S2の1番目なので別に計算しない）。
"""
import re, json, time, html

MAX_DEFS, MAX_CHARS = 8, 220


def clean(s):
    s = re.sub(r"<[^>]+>", "", s or "")
    s = html.unescape(s)
    return re.sub(r"\s+", " ", s).strip()[:MAX_CHARS]


def parse_wiktionary(js):
    """Wiktionary REST（page/definition）の答えから、英語の名詞の意味だけを取り出す。"""
    out = []
    for blk in (js or {}).get("en", []):
        if blk.get("partOfSpeech") != "Noun":
            continue
        for d in blk.get("definitions", []):
            t = clean(d.get("definition", ""))
            if t and not t.lower().startswith(("plural of", "alternative form of", "alternative spelling of", "obsolete form of")):
                out.append(t)
    return out[:MAX_DEFS]


def strip_wikitext(t):
    t = re.sub(r"\{\{(?:lb|lbl|label|q|qualifier|i|gloss)\|[^{}]*\}\}", "", t)        # ラベル類は消す
    for _ in range(3):
        t = re.sub(r"\{\{[^{}|]*\|(?:[^{}|]*\|)*([^{}|=]*)\}\}", r"\1", t)          # {{l|en|bar}} → bar
    t = re.sub(r"\{\{[^{}]*\}\}", "", t)
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)                           # [[a|b]] → b
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    return clean(t)


SKIP = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of")


def parse_wikitext(wt):
    """予備：MediaWikiの wikitext から英語の名詞の意味（'# ' の行）を取り出す。"""
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return []
    out = []
    for sec in re.finditer(r"(?ms)^===+\s*Noun\s*===+\s*$(.*?)(?=^===|\Z)", m.group(1)):
        for line in sec.group(1).splitlines():
            if line.startswith("# "):
                raw = line[2:].strip()
                if re.match(r"\{\{(plural of|alternative form of|alt form|alternative spelling of|obsolete form of)\|", raw):
                    continue
                d = strip_wikitext(raw)
                if d and not d.lower().startswith(SKIP):
                    out.append(d)
    return out[:MAX_DEFS]


def fetch_all(words, get_json, get_wikitext=None, log=print, sleep=0.1):
    """get_json(word) -> dict か None。空なら予備として get_wikitext(word) -> str。取れなかった語は空のリスト。"""
    res = {}
    for i, w in enumerate(words):
        try:
            res[w] = parse_wiktionary(get_json(w))
        except Exception as e:  # 取れない語は空
            res[w] = []
        if not res[w] and get_wikitext is not None:
            try:
                res[w] = parse_wikitext(get_wikitext(w))
            except Exception:
                pass
        if sleep:
            time.sleep(sleep)
        if (i + 1) % 100 == 0:
            log(f"Wiktionary {i + 1}/{len(words)}")
    return res


def sources(it):
    wk = it.get("wk_defs") or []
    return {"S1": it["wn_defs"][:MAX_DEFS], "S2": wk, "S3": wk[:1]}


def gemma_question(it, defs):
    q = f"Is the following statement true? Answer Yes or No only.\nStatement: {it['stmt']}"
    if defs is None:
        return q
    if not defs:
        return q  # 説明文がない語は、説明文なしと同じ聞き方（結果に has_defs を残す）
    lines = "\n".join(f"{k + 1}. {d}" for k, d in enumerate(defs))
    return f"Dictionary definitions of the word \"{it['ent']}\":\n{lines}\n\n" + q


def premise(it, d):
    return f"\"{it['ent']}\" means: {d}"


def run(items, gemma_logits, nli_probs, log=print):
    """gemma_logits(list[str]) -> list[float]（はい−いいえ）。nli_probs(premises, hypotheses) -> list[[含む,中立,矛盾]]"""
    # Gemma：説明文なし＋S1〜S3
    for key in ["none", "S1", "S2", "S3"]:
        qs = [gemma_question(it, None if key == "none" else sources(it)[key]) for it in items]
        L = gemma_logits(qs)
        for it, v in zip(items, L):
            it.setdefault("gemma", {})[key] = float(v)
        log(f"Gemma {key} 済み")
    # NLI：意味ごと（S1, S2）
    for key in ["S1", "S2"]:
        P, H, idx = [], [], []
        for i, it in enumerate(items):
            for d in sources(it)[key]:
                P.append(premise(it, d)); H.append(it["stmt"]); idx.append(i)
        out = nli_probs(P, H) if P else []
        for it in items:
            it.setdefault("nli", {})[key] = []
        for i, p in zip(idx, out):
            items[i]["nli"][key].append([float(x) for x in p])
        log(f"NLI {key} 済み（{len(P)}組）")
    return items


def mock(items, seed=0):
    import random
    r = random.Random(seed)
    for it in items:
        it["wk_defs"] = [f"mock sense {k} of {it['ent']}" for k in range(r.randint(0, 4))]
    gl = lambda qs: [r.gauss(0, 3) for _ in qs]
    def nl(P, H):
        out = []
        for _ in P:
            a = [r.random() for _ in range(3)]; s = sum(a); out.append([x / s for x in a])
        return out
    return run(items, gl, nl)

In [ ]:
ITEMS = json.loads(r'''[{"id": 0, "ent": "barbell", "stmt": "Barbell is a type of tool.", "wn_defs": ["a bar to which heavy discs are attached at each end; used in weightlifting"]}, {"id": 1, "ent": "tenement", "stmt": "Tenement is a type of building.", "wn_defs": ["a run-down apartment house barely meeting minimal standards"]}, {"id": 2, "ent": "porpoise", "stmt": "Porpoise is a type of dog.", "wn_defs": ["any of several small gregarious cetacean mammals having a blunt snout and many teeth"]}, {"id": 3, "ent": "juvenile", "stmt": "Juvenile is a type of boat.", "wn_defs": ["a young person, not fully developed"]}, {"id": 4, "ent": "barbell", "stmt": "Barbell is a type of mammal.", "wn_defs": ["a bar to which heavy discs are attached at each end; used in weightlifting"]}, {"id": 5, "ent": "grocer", "stmt": "Grocer is a type of living thing.", "wn_defs": ["a retail merchant who sells foodstuffs (and some household supplies)"]}, {"id": 6, "ent": "gramophone", "stmt": "Gramophone is a type of game.", "wn_defs": ["an antique record player; the sound of the vibrating needle is amplified acoustically"]}, {"id": 7, "ent": "hooch", "stmt": "Hooch is a type of artist.", "wn_defs": ["an illicitly distilled (and usually inferior) alcoholic liquor"]}, {"id": 8, "ent": "bilingual", "stmt": "Bilingual is a type of person.", "wn_defs": ["a person who speaks two languages fluently"]}, {"id": 9, "ent": "minnow", "stmt": "Minnow is a type of disease.", "wn_defs": ["very small European freshwater fish common in gravelly streams"]}, {"id": 10, "ent": "gazelle", "stmt": "Gazelle is a type of food.", "wn_defs": ["small swift graceful antelope of Africa and Asia having lustrous eyes"]}, {"id": 11, "ent": "gasket", "stmt": "Gasket is a type of game.", "wn_defs": ["seal consisting of a ring for packing pistons or sealing a pipe joint"]}, {"id": 12, "ent": "isolationist", "stmt": "Isolationist is a type of living thing.", "wn_defs": ["an advocate of isolationism in international affairs"]}, {"id": 13, "ent": "clamp", "stmt": "Clamp is a type of tool.", "wn_defs": ["a device (generally used by carpenters) that holds things firmly together"]}, {"id": 14, "ent": "copycat", "stmt": "Copycat is a type of body part.", "wn_defs": ["someone who copies the words or behavior of another"]}, {"id": 15, "ent": "peshmerga", "stmt": "Peshmerga is a type of living thing.", "wn_defs": ["a member of a Kurdish guerilla organization that fights for a free Kurdish state"]}, {"id": 16, "ent": "tabor", "stmt": "Tabor is a type of boat.", "wn_defs": ["a small drum with one head of soft calfskin"]}, {"id": 17, "ent": "stepmother", "stmt": "Stepmother is a type of person.", "wn_defs": ["the wife of your father by a subsequent marriage"]}, {"id": 18, "ent": "terrier", "stmt": "Terrier is a type of bird.", "wn_defs": ["any of several usually small short-bodied breeds originally trained to hunt animals living underground"]}, {"id": 19, "ent": "preteen", "stmt": "Preteen is a type of metal.", "wn_defs": ["a preadolescent boy or girl (usually between 9 and 12 years of age)"]}, {"id": 20, "ent": "composer", "stmt": "Composer is a type of person.", "wn_defs": ["someone who composes music as a profession"]}, {"id": 21, "ent": "suitor", "stmt": "Suitor is a type of living thing.", "wn_defs": ["a man who courts a woman"]}, {"id": 22, "ent": "leaker", "stmt": "Leaker is a type of person.", "wn_defs": ["a surreptitious informant"]}, {"id": 23, "ent": "actuator", "stmt": "Actuator is a type of machine.", "wn_defs": ["a mechanism that puts something into automatic action"]}, {"id": 24, "ent": "lally", "stmt": "Lally is a type of tool.", "wn_defs": ["support column consisting of a steel cylinder filled with concrete"]}, {"id": 25, "ent": "epileptic", "stmt": "Epileptic is a type of disease.", "wn_defs": ["a person who has epilepsy"]}, {"id": 26, "ent": "pew", "stmt": "Pew is a type of man-made object.", "wn_defs": ["long bench with backs; used in church by the congregation"]}, {"id": 27, "ent": "microphone", "stmt": "Microphone is a type of equipment.", "wn_defs": ["device for converting sound waves into electrical energy"]}, {"id": 28, "ent": "felt", "stmt": "Felt is a type of material.", "wn_defs": ["a fabric made of compressed matted animal fibers"]}, {"id": 29, "ent": "primrose", "stmt": "Primrose is a type of living thing.", "wn_defs": ["any of numerous short-stemmed plants of the genus Primula having tufted basal leaves and showy flowers clustered in umbels or heads"]}, {"id": 30, "ent": "quartermaster", "stmt": "Quartermaster is a type of person.", "wn_defs": ["an army officer who provides clothing and subsistence for troops"]}, {"id": 31, "ent": "delegate", "stmt": "Delegate is a type of person.", "wn_defs": ["a person appointed or elected to represent others"]}, {"id": 32, "ent": "grubby", "stmt": "Grubby is a type of toy.", "wn_defs": ["small sculpin of the coast of New England"]}, {"id": 33, "ent": "prince", "stmt": "Prince is a type of artist.", "wn_defs": ["a male member of a royal family other than the sovereign (especially the son of a sovereign)"]}, {"id": 34, "ent": "locomotive", "stmt": "Locomotive is a type of man-made object.", "wn_defs": ["a wheeled vehicle consisting of a self-propelled engine that is used to draw trains along railway tracks"]}, {"id": 35, "ent": "lobbyist", "stmt": "Lobbyist is a type of living thing.", "wn_defs": ["someone who is employed to persuade legislators to vote for legislation that favors the lobbyist's employer"]}, {"id": 36, "ent": "workman", "stmt": "Workman is a type of footwear.", "wn_defs": ["an employee who performs manual or industrial labor"]}, {"id": 37, "ent": "stupid", "stmt": "Stupid is a type of person.", "wn_defs": ["a person who is not very bright"]}, {"id": 38, "ent": "watchman", "stmt": "Watchman is a type of body part.", "wn_defs": ["a guard who keeps watch"]}, {"id": 39, "ent": "headliner", "stmt": "Headliner is a type of equipment.", "wn_defs": ["a performer who receives prominent billing"]}, {"id": 40, "ent": "geranium", "stmt": "Geranium is a type of living thing.", "wn_defs": ["any of numerous plants of the family Geraniaceae"]}, {"id": 41, "ent": "guerrilla", "stmt": "Guerrilla is a type of weapon.", "wn_defs": ["a member of an irregular armed force that fights a stronger force by sabotage and harassment"]}, {"id": 42, "ent": "sheeting", "stmt": "Sheeting is a type of food.", "wn_defs": ["fabric from which bed sheets are made"]}, {"id": 43, "ent": "consulate", "stmt": "Consulate is a type of device.", "wn_defs": ["diplomatic building that serves as the residence or workplace of a consul"]}, {"id": 44, "ent": "dinky", "stmt": "Dinky is a type of artist.", "wn_defs": ["a small locomotive"]}, {"id": 45, "ent": "caterer", "stmt": "Caterer is a type of sport.", "wn_defs": ["someone who provides food and service (as for a party)"]}, {"id": 46, "ent": "rickey", "stmt": "Rickey is a type of tree.", "wn_defs": ["a mixed drink made of sweetened lime juice and soda water usually with liquor"]}, {"id": 47, "ent": "headmistress", "stmt": "Headmistress is a type of worker.", "wn_defs": ["a woman headmaster"]}, {"id": 48, "ent": "educator", "stmt": "Educator is a type of worker.", "wn_defs": ["someone who educates young people"]}, {"id": 49, "ent": "golf", "stmt": "Golf is a type of metal.", "wn_defs": ["a game played on a large open course with 9 or 18 holes; the object is use as few strokes as possible in playing all the holes"]}, {"id": 50, "ent": "velvet", "stmt": "Velvet is a type of man-made object.", "wn_defs": ["a silky densely piled fabric with a plain back"]}, {"id": 51, "ent": "biographer", "stmt": "Biographer is a type of disease.", "wn_defs": ["someone who writes an account of a person's life"]}, {"id": 52, "ent": "zoologist", "stmt": "Zoologist is a type of living thing.", "wn_defs": ["a specialist in the branch of biology dealing with animals"]}, {"id": 53, "ent": "doge", "stmt": "Doge is a type of living thing.", "wn_defs": ["formerly the chief magistrate in the republics of Venice and Genoa"]}, {"id": 54, "ent": "mimic", "stmt": "Mimic is a type of reptile.", "wn_defs": ["someone who mimics (especially an actor or actress)"]}, {"id": 55, "ent": "passerby", "stmt": "Passerby is a type of organ.", "wn_defs": ["a person who passes by casually or by chance"]}, {"id": 56, "ent": "dieter", "stmt": "Dieter is a type of fish.", "wn_defs": ["a person who diets"]}, {"id": 57, "ent": "niqab", "stmt": "Niqab is a type of reptile.", "wn_defs": ["a face veil covering the lower part of the face (up to the eyes) worn by observant Muslim women"]}, {"id": 58, "ent": "bulldozer", "stmt": "Bulldozer is a type of machine.", "wn_defs": ["large powerful tractor; a large blade in front flattens areas of ground"]}, {"id": 59, "ent": "deflector", "stmt": "Deflector is a type of equipment.", "wn_defs": ["a device intended to turn aside the flow of something (water or air or smoke etc)"]}, {"id": 60, "ent": "schoolgirl", "stmt": "Schoolgirl is a type of metal.", "wn_defs": ["a girl attending school"]}, {"id": 61, "ent": "falconer", "stmt": "Falconer is a type of sport.", "wn_defs": ["a person who breeds and trains hawks and who follows the sport of falconry"]}, {"id": 62, "ent": "marlin", "stmt": "Marlin is a type of drink.", "wn_defs": ["large long-jawed oceanic sport fishes; related to sailfishes and spearfishes; not completely cold-blooded i.e. able to warm their brains and eyes"]}, {"id": 63, "ent": "plumber", "stmt": "Plumber is a type of bird.", "wn_defs": ["a craftsman who installs and repairs pipes and fixtures and appliances"]}, {"id": 64, "ent": "sarcophagus", "stmt": "Sarcophagus is a type of man-made object.", "wn_defs": ["a stone coffin (usually bearing sculpture or inscriptions)"]}, {"id": 65, "ent": "amputee", "stmt": "Amputee is a type of person.", "wn_defs": ["someone who has had a limb removed by amputation"]}, {"id": 66, "ent": "kimono", "stmt": "Kimono is a type of fish.", "wn_defs": ["a loose robe; imitated from robes originally worn by Japanese"]}, {"id": 67, "ent": "temp", "stmt": "Temp is a type of worker.", "wn_defs": ["a worker (especially in an office) hired on a temporary basis"]}, {"id": 68, "ent": "propeller", "stmt": "Propeller is a type of reptile.", "wn_defs": ["a mechanical device that rotates to push against air or water"]}, {"id": 69, "ent": "wombat", "stmt": "Wombat is a type of building.", "wn_defs": ["burrowing herbivorous Australian marsupials about the size of a badger"]}, {"id": 70, "ent": "paintbrush", "stmt": "Paintbrush is a type of tool.", "wn_defs": ["a brush used as an applicator (to apply paint)"]}, {"id": 71, "ent": "womanizer", "stmt": "Womanizer is a type of body part.", "wn_defs": ["a man who likes many women and has short sexual relationships with them"]}, {"id": 72, "ent": "greenhouse", "stmt": "Greenhouse is a type of man-made object.", "wn_defs": ["a building with glass walls and roof; for the cultivation and exhibition of plants under controlled conditions"]}, {"id": 73, "ent": "tactician", "stmt": "Tactician is a type of food.", "wn_defs": ["a person who is skilled at planning tactics"]}, {"id": 74, "ent": "achiever", "stmt": "Achiever is a type of fruit.", "wn_defs": ["a person with a record of successes"]}, {"id": 75, "ent": "moll", "stmt": "Moll is a type of person.", "wn_defs": ["the girlfriend of a gangster"]}, {"id": 76, "ent": "bonsai", "stmt": "Bonsai is a type of plant.", "wn_defs": ["a dwarfed ornamental tree or shrub grown in a tray or shallow pot"]}, {"id": 77, "ent": "underdog", "stmt": "Underdog is a type of tree.", "wn_defs": ["one at a disadvantage and expected to lose"]}, {"id": 78, "ent": "loom", "stmt": "Loom is a type of car.", "wn_defs": ["a textile machine for weaving yarn into a textile"]}, {"id": 79, "ent": "individualist", "stmt": "Individualist is a type of person.", "wn_defs": ["a person who pursues independent thought or action"]}, {"id": 80, "ent": "easel", "stmt": "Easel is a type of sport.", "wn_defs": ["an upright tripod for displaying something (usually an artist's canvas)"]}, {"id": 81, "ent": "finalist", "stmt": "Finalist is a type of athlete.", "wn_defs": ["a contestant who reaches the final stages of a competition"]}, {"id": 82, "ent": "sagebrush", "stmt": "Sagebrush is a type of weapon.", "wn_defs": ["any of several North American composite subshrubs of the genera Artemis or Seriphidium"]}, {"id": 83, "ent": "uniform", "stmt": "Uniform is a type of equipment.", "wn_defs": ["clothing of distinctive design worn by members of a particular group as a means of identification"]}, {"id": 84, "ent": "bonsai", "stmt": "Bonsai is a type of animal.", "wn_defs": ["a dwarfed ornamental tree or shrub grown in a tray or shallow pot"]}, {"id": 85, "ent": "receptionist", "stmt": "Receptionist is a type of worker.", "wn_defs": ["a secretary whose main duty is to answer the telephone and receive visitors"]}, {"id": 86, "ent": "commoner", "stmt": "Commoner is a type of worker.", "wn_defs": ["a person who holds no title"]}, {"id": 87, "ent": "modernist", "stmt": "Modernist is a type of person.", "wn_defs": ["an artist who makes a deliberate break with previous styles"]}, {"id": 88, "ent": "cuppa", "stmt": "Cuppa is a type of container.", "wn_defs": ["a cup of tea"]}, {"id": 89, "ent": "cornerback", "stmt": "Cornerback is a type of athlete.", "wn_defs": ["a defensive football player stationed outside the linebackers"]}, {"id": 90, "ent": "minnow", "stmt": "Minnow is a type of game.", "wn_defs": ["very small European freshwater fish common in gravelly streams"]}, {"id": 91, "ent": "neurologist", "stmt": "Neurologist is a type of worker.", "wn_defs": ["a medical specialist in the nervous system and the disorders affecting it"]}, {"id": 92, "ent": "kitten", "stmt": "Kitten is a type of mammal.", "wn_defs": ["young domestic cat"]}, {"id": 93, "ent": "undercarriage", "stmt": "Undercarriage is a type of device.", "wn_defs": ["framework that serves as a support for the body of a vehicle"]}, {"id": 94, "ent": "reporter", "stmt": "Reporter is a type of worker.", "wn_defs": ["a person who investigates and reports or edits news stories"]}, {"id": 95, "ent": "critter", "stmt": "Critter is a type of equipment.", "wn_defs": ["a regional term for `creature' (especially for domestic animals)"]}, {"id": 96, "ent": "granddaughter", "stmt": "Granddaughter is a type of mammal.", "wn_defs": ["a female grandchild"]}, {"id": 97, "ent": "crusher", "stmt": "Crusher is a type of machine.", "wn_defs": ["a device that crushes something"]}, {"id": 98, "ent": "accomplice", "stmt": "Accomplice is a type of athlete.", "wn_defs": ["a person who joins with another in carrying out some plan (especially an unethical or illegal plan)"]}, {"id": 99, "ent": "dubbing", "stmt": "Dubbing is a type of device.", "wn_defs": ["a new soundtrack that is added to a film"]}, {"id": 100, "ent": "tortoise", "stmt": "Tortoise is a type of reptile.", "wn_defs": ["usually herbivorous land turtles having clawed elephant-like limbs; worldwide in arid area except Australia and Antarctica"]}, {"id": 101, "ent": "jukebox", "stmt": "Jukebox is a type of equipment.", "wn_defs": ["a cabinet containing an automatic record player; records are played by inserting a coin"]}, {"id": 102, "ent": "restaurant", "stmt": "Restaurant is a type of building.", "wn_defs": ["a building where people go to eat"]}, {"id": 103, "ent": "sectional", "stmt": "Sectional is a type of man-made object.", "wn_defs": ["a piece of furniture made up of sections that can be arranged individually or together"]}, {"id": 104, "ent": "heifer", "stmt": "Heifer is a type of living thing.", "wn_defs": ["young cow"]}, {"id": 105, "ent": "kea", "stmt": "Kea is a type of living thing.", "wn_defs": ["large brownish-green New Zealand parrot"]}, {"id": 106, "ent": "minion", "stmt": "Minion is a type of living thing.", "wn_defs": ["a servile or fawning dependant"]}, {"id": 107, "ent": "opportunist", "stmt": "Opportunist is a type of person.", "wn_defs": ["a person who places expediency above principle"]}, {"id": 108, "ent": "conformist", "stmt": "Conformist is a type of artist.", "wn_defs": ["someone who conforms to established standards of conduct (especially in religious matters)"]}, {"id": 109, "ent": "burglar", "stmt": "Burglar is a type of body part.", "wn_defs": ["a thief who enters a building with intent to steal"]}, {"id": 110, "ent": "buddy", "stmt": "Buddy is a type of man-made object.", "wn_defs": ["a close friend who accompanies his buddies in their activities"]}, {"id": 111, "ent": "wraparound", "stmt": "Wraparound is a type of metal.", "wn_defs": ["a garment (as a dress or coat) with a full length opening; adjusts to the body by wrapping around"]}, {"id": 112, "ent": "vise", "stmt": "Vise is a type of tool.", "wn_defs": ["a holding device attached to a workbench; has two jaws to hold workpiece firmly in place"]}, {"id": 113, "ent": "helicopter", "stmt": "Helicopter is a type of man-made object.", "wn_defs": ["an aircraft without wings that obtains its lift from the rotation of overhead blades"]}, {"id": 114, "ent": "calender", "stmt": "Calender is a type of tool.", "wn_defs": ["a machine that smooths or glazes paper or cloth by pressing it between plates or passing it through rollers"]}, {"id": 115, "ent": "snob", "stmt": "Snob is a type of game.", "wn_defs": ["a person regarded as arrogant and annoying"]}, {"id": 116, "ent": "drunkard", "stmt": "Drunkard is a type of living thing.", "wn_defs": ["a chronic drinker"]}, {"id": 117, "ent": "bandicoot", "stmt": "Bandicoot is a type of disease.", "wn_defs": ["any of various agile ratlike terrestrial marsupials of Australia and adjacent islands; insectivorous and herbivorous"]}, {"id": 118, "ent": "musket", "stmt": "Musket is a type of vehicle.", "wn_defs": ["a muzzle-loading shoulder gun with a long barrel; formerly used by infantrymen"]}, {"id": 119, "ent": "grader", "stmt": "Grader is a type of sport.", "wn_defs": ["a judge who assigns grades to something"]}, {"id": 120, "ent": "briefcase", "stmt": "Briefcase is a type of vegetable.", "wn_defs": ["a case with a handle; for carrying papers or files or books"]}, {"id": 121, "ent": "alternator", "stmt": "Alternator is a type of vehicle.", "wn_defs": ["an old term for an electric generator that produces alternating current (especially in automobiles)"]}, {"id": 122, "ent": "wallaby", "stmt": "Wallaby is a type of plant.", "wn_defs": ["any of various small or medium-sized kangaroos; often brightly colored"]}, {"id": 123, "ent": "interceptor", "stmt": "Interceptor is a type of aircraft.", "wn_defs": ["a fast maneuverable fighter plane designed to intercept enemy aircraft"]}, {"id": 124, "ent": "carbine", "stmt": "Carbine is a type of equipment.", "wn_defs": ["light automatic rifle"]}, {"id": 125, "ent": "spitfire", "stmt": "Spitfire is a type of living thing.", "wn_defs": ["a highly emotional and quick-tempered person (especially a girl or woman)"]}, {"id": 126, "ent": "huntress", "stmt": "Huntress is a type of worker.", "wn_defs": ["a woman hunter"]}, {"id": 127, "ent": "heathen", "stmt": "Heathen is a type of flower.", "wn_defs": ["a person who does not acknowledge your god"]}, {"id": 128, "ent": "senna", "stmt": "Senna is a type of man-made object.", "wn_defs": ["any of various plants of the genus Senna having pinnately compound leaves and showy usually yellow flowers; many are used medicinally"]}, {"id": 129, "ent": "macho", "stmt": "Macho is a type of animal.", "wn_defs": ["a male exhibiting or characterized by machismo"]}, {"id": 130, "ent": "beloved", "stmt": "Beloved is a type of insect.", "wn_defs": ["a beloved person; used as terms of endearment"]}, {"id": 131, "ent": "wallaby", "stmt": "Wallaby is a type of living thing.", "wn_defs": ["any of various small or medium-sized kangaroos; often brightly colored"]}, {"id": 132, "ent": "accountant", "stmt": "Accountant is a type of game.", "wn_defs": ["someone who maintains and audits business accounts"]}, {"id": 133, "ent": "shogun", "stmt": "Shogun is a type of fabric.", "wn_defs": ["a hereditary military dictator of Japan; the shoguns ruled Japan until the revolution of 1867-68"]}, {"id": 134, "ent": "parsonage", "stmt": "Parsonage is a type of boat.", "wn_defs": ["an official residence provided by a church for its parson or vicar or rector"]}, {"id": 135, "ent": "timepiece", "stmt": "Timepiece is a type of tool.", "wn_defs": ["a measuring instrument or device for keeping time"]}, {"id": 136, "ent": "biter", "stmt": "Biter is a type of living thing.", "wn_defs": ["someone who bites"]}, {"id": 137, "ent": "rugby", "stmt": "Rugby is a type of equipment.", "wn_defs": ["a form of football played with an oval ball"]}, {"id": 138, "ent": "bedroom", "stmt": "Bedroom is a type of building.", "wn_defs": ["a room used primarily for sleeping"]}, {"id": 139, "ent": "wombat", "stmt": "Wombat is a type of tree.", "wn_defs": ["burrowing herbivorous Australian marsupials about the size of a badger"]}, {"id": 140, "ent": "everlasting", "stmt": "Everlasting is a type of game.", "wn_defs": ["any of various plants of various genera of the family Compositae having flowers that can be dried without loss of form or color"]}, {"id": 141, "ent": "auctioneer", "stmt": "Auctioneer is a type of material.", "wn_defs": ["an agent who conducts an auction"]}, {"id": 142, "ent": "theorist", "stmt": "Theorist is a type of person.", "wn_defs": ["someone who theorizes (especially in science or art)"]}, {"id": 143, "ent": "skater", "stmt": "Skater is a type of person.", "wn_defs": ["someone who skates"]}, {"id": 144, "ent": "mariner", "stmt": "Mariner is a type of metal.", "wn_defs": ["a man who serves as a sailor"]}, {"id": 145, "ent": "freelancer", "stmt": "Freelancer is a type of fish.", "wn_defs": ["a writer or artist who sells services to different employers without a long-term contract with any of them"]}, {"id": 146, "ent": "dubbing", "stmt": "Dubbing is a type of game.", "wn_defs": ["a new soundtrack that is added to a film"]}, {"id": 147, "ent": "simulator", "stmt": "Simulator is a type of game.", "wn_defs": ["a machine that simulates an environment for the purpose of training or research"]}, {"id": 148, "ent": "circuitry", "stmt": "Circuitry is a type of sport.", "wn_defs": ["electronic equipment consisting of a system of circuits"]}, {"id": 149, "ent": "galleon", "stmt": "Galleon is a type of boat.", "wn_defs": ["a large square-rigged sailing ship with three or more masts; used by the Spanish for commerce and war from the 15th to 18th centuries"]}, {"id": 150, "ent": "steed", "stmt": "Steed is a type of animal.", "wn_defs": ["(literary) a spirited horse for state or war"]}, {"id": 151, "ent": "mosquito", "stmt": "Mosquito is a type of weapon.", "wn_defs": ["two-winged insect whose female has a long proboscis to pierce the skin and suck the blood of humans and animals"]}, {"id": 152, "ent": "legate", "stmt": "Legate is a type of game.", "wn_defs": ["a member of a legation"]}, {"id": 153, "ent": "maxi", "stmt": "Maxi is a type of room.", "wn_defs": ["a long skirt ending below the calf"]}, {"id": 154, "ent": "bodybuilder", "stmt": "Bodybuilder is a type of animal.", "wn_defs": ["someone who does special exercises to develop a brawny musculature"]}, {"id": 155, "ent": "newlywed", "stmt": "Newlywed is a type of living thing.", "wn_defs": ["someone recently married"]}, {"id": 156, "ent": "finch", "stmt": "Finch is a type of material.", "wn_defs": ["any of numerous small songbirds with short stout bills adapted for crushing seeds"]}, {"id": 157, "ent": "flea", "stmt": "Flea is a type of vegetable.", "wn_defs": ["any wingless bloodsucking parasitic insect noted for ability to leap"]}, {"id": 158, "ent": "betel", "stmt": "Betel is a type of tree.", "wn_defs": ["Asian pepper plant whose dried leaves are chewed with betel nut (seed of the betel palm) by southeast Asians"]}, {"id": 159, "ent": "microphone", "stmt": "Microphone is a type of sport.", "wn_defs": ["device for converting sound waves into electrical energy"]}, {"id": 160, "ent": "airliner", "stmt": "Airliner is a type of boat.", "wn_defs": ["a commercial airplane that carries passengers"]}, {"id": 161, "ent": "commodore", "stmt": "Commodore is a type of machine.", "wn_defs": ["a commissioned naval officer who ranks above a captain and below a rear admiral; the lowest grade of admiral"]}, {"id": 162, "ent": "billionaire", "stmt": "Billionaire is a type of body part.", "wn_defs": ["a very rich person whose material wealth is valued at more than a billion dollars"]}, {"id": 163, "ent": "roadhouse", "stmt": "Roadhouse is a type of game.", "wn_defs": ["an inn (usually outside city limits on a main road) providing meals and liquor and dancing and (sometimes) gambling"]}, {"id": 164, "ent": "hater", "stmt": "Hater is a type of person.", "wn_defs": ["a person who hates"]}, {"id": 165, "ent": "marc", "stmt": "Marc is a type of fruit.", "wn_defs": ["made from residue of grapes or apples after pressing"]}, {"id": 166, "ent": "tugboat", "stmt": "Tugboat is a type of disease.", "wn_defs": ["a powerful small boat designed to pull or push larger ships"]}, {"id": 167, "ent": "blouse", "stmt": "Blouse is a type of fish.", "wn_defs": ["a top worn by women"]}, {"id": 168, "ent": "houseboat", "stmt": "Houseboat is a type of device.", "wn_defs": ["a barge that is designed and equipped for use as a dwelling"]}, {"id": 169, "ent": "bovine", "stmt": "Bovine is a type of toy.", "wn_defs": ["any of various members of the genus Bos"]}, {"id": 170, "ent": "porpoise", "stmt": "Porpoise is a type of athlete.", "wn_defs": ["any of several small gregarious cetacean mammals having a blunt snout and many teeth"]}, {"id": 171, "ent": "everlasting", "stmt": "Everlasting is a type of plant.", "wn_defs": ["any of various plants of various genera of the family Compositae having flowers that can be dried without loss of form or color"]}, {"id": 172, "ent": "tomahawk", "stmt": "Tomahawk is a type of man-made object.", "wn_defs": ["weapon consisting of a fighting ax; used by North American Indians"]}, {"id": 173, "ent": "national", "stmt": "National is a type of boat.", "wn_defs": ["a person who owes allegiance to that nation"]}, {"id": 174, "ent": "underpants", "stmt": "Underpants is a type of man-made object.", "wn_defs": ["an undergarment that covers the body from the waist no further than to the thighs; usually worn next to the skin"]}, {"id": 175, "ent": "hitchhiker", "stmt": "Hitchhiker is a type of worker.", "wn_defs": ["a person who travels by getting free rides from passing vehicles"]}, {"id": 176, "ent": "cutthroat", "stmt": "Cutthroat is a type of plant.", "wn_defs": ["someone who murders by cutting the victim's throat"]}, {"id": 177, "ent": "theologian", "stmt": "Theologian is a type of reptile.", "wn_defs": ["someone who is learned in theology or who speculates about theology"]}, {"id": 178, "ent": "facilitator", "stmt": "Facilitator is a type of vegetable.", "wn_defs": ["someone who makes progress easier"]}, {"id": 179, "ent": "rickey", "stmt": "Rickey is a type of man-made object.", "wn_defs": ["a mixed drink made of sweetened lime juice and soda water usually with liquor"]}, {"id": 180, "ent": "songwriter", "stmt": "Songwriter is a type of dish.", "wn_defs": ["a composer of words or music for popular songs"]}, {"id": 181, "ent": "extremist", "stmt": "Extremist is a type of fruit.", "wn_defs": ["a person who holds extreme views"]}, {"id": 182, "ent": "transvestite", "stmt": "Transvestite is a type of sport.", "wn_defs": ["someone who adopts the dress or manner or sexual role of the opposite sex"]}, {"id": 183, "ent": "warship", "stmt": "Warship is a type of fruit.", "wn_defs": ["a government ship that is available for waging war"]}, {"id": 184, "ent": "spokeswoman", "stmt": "Spokeswoman is a type of body part.", "wn_defs": ["a female spokesperson"]}, {"id": 185, "ent": "brill", "stmt": "Brill is a type of animal.", "wn_defs": ["European food fish"]}, {"id": 186, "ent": "secessionist", "stmt": "Secessionist is a type of metal.", "wn_defs": ["an advocate of secessionism"]}, {"id": 187, "ent": "beloved", "stmt": "Beloved is a type of fabric.", "wn_defs": ["a beloved person; used as terms of endearment"]}, {"id": 188, "ent": "impala", "stmt": "Impala is a type of sport.", "wn_defs": ["African antelope with ridged curved horns; moves with enormous leaps"]}, {"id": 189, "ent": "firearm", "stmt": "Firearm is a type of man-made object.", "wn_defs": ["a portable gun"]}, {"id": 190, "ent": "mathematician", "stmt": "Mathematician is a type of worker.", "wn_defs": ["a person skilled in mathematics"]}, {"id": 191, "ent": "buttercup", "stmt": "Buttercup is a type of man-made object.", "wn_defs": ["any of various plants of the genus Ranunculus"]}, {"id": 192, "ent": "notary", "stmt": "Notary is a type of game.", "wn_defs": ["someone legally empowered to witness signatures and certify a document's validity and to take depositions"]}, {"id": 193, "ent": "missus", "stmt": "Missus is a type of boat.", "wn_defs": ["informal term of address for someone's wife"]}, {"id": 194, "ent": "freelancer", "stmt": "Freelancer is a type of living thing.", "wn_defs": ["a writer or artist who sells services to different employers without a long-term contract with any of them"]}, {"id": 195, "ent": "beret", "stmt": "Beret is a type of toy.", "wn_defs": ["a cap with no brim or bill; made of soft cloth"]}, {"id": 196, "ent": "georgette", "stmt": "Georgette is a type of clothing.", "wn_defs": ["a thin silk dress material"]}, {"id": 197, "ent": "parsonage", "stmt": "Parsonage is a type of man-made object.", "wn_defs": ["an official residence provided by a church for its parson or vicar or rector"]}, {"id": 198, "ent": "dongle", "stmt": "Dongle is a type of metal.", "wn_defs": ["(computer science) an electronic device that must be attached to a computer in order for it to use protected software"]}, {"id": 199, "ent": "spacecraft", "stmt": "Spacecraft is a type of vehicle.", "wn_defs": ["a craft capable of traveling in outer space; technically, a satellite around the sun"]}, {"id": 200, "ent": "eunuch", "stmt": "Eunuch is a type of living thing.", "wn_defs": ["a man who has been castrated and is incapable of reproduction"]}, {"id": 201, "ent": "apparel", "stmt": "Apparel is a type of clothing.", "wn_defs": ["clothing in general"]}, {"id": 202, "ent": "messenger", "stmt": "Messenger is a type of person.", "wn_defs": ["a person who carries a message"]}, {"id": 203, "ent": "strainer", "stmt": "Strainer is a type of game.", "wn_defs": ["a filter to retain larger pieces while smaller pieces and liquids pass through"]}, {"id": 204, "ent": "bookkeeper", "stmt": "Bookkeeper is a type of living thing.", "wn_defs": ["someone who records the transactions of a business"]}, {"id": 205, "ent": "clamp", "stmt": "Clamp is a type of musical instrument.", "wn_defs": ["a device (generally used by carpenters) that holds things firmly together"]}, {"id": 206, "ent": "sieve", "stmt": "Sieve is a type of body part.", "wn_defs": ["a strainer for separating lumps from powdered material or grading particles"]}, {"id": 207, "ent": "madame", "stmt": "Madame is a type of insect.", "wn_defs": ["title used for a married Frenchwoman"]}, {"id": 208, "ent": "bethel", "stmt": "Bethel is a type of body part.", "wn_defs": ["a house of worship (especially one for sailors)"]}, {"id": 209, "ent": "moray", "stmt": "Moray is a type of disease.", "wn_defs": ["family of brightly colored voracious eels of warm coastal waters; generally nonaggressive to humans but larger species are dangerous if provoked"]}, {"id": 210, "ent": "lemur", "stmt": "Lemur is a type of animal.", "wn_defs": ["large-eyed arboreal prosimian having foxy faces and long furry tails"]}, {"id": 211, "ent": "orangutan", "stmt": "Orangutan is a type of fruit.", "wn_defs": ["large long-armed ape of Borneo and Sumatra having arboreal habits"]}, {"id": 212, "ent": "athlete", "stmt": "Athlete is a type of living thing.", "wn_defs": ["a person trained to compete in sports"]}, {"id": 213, "ent": "vole", "stmt": "Vole is a type of worker.", "wn_defs": ["any of various small mouselike rodents of the family Cricetidae (especially of genus Microtus) having a stout short-tailed body and inconspicuous ears and inhabiting fields or meadows"]}, {"id": 214, "ent": "peony", "stmt": "Peony is a type of car.", "wn_defs": ["any of numerous plants widely cultivated for their showy single or double red or pink or white flowers"]}, {"id": 215, "ent": "undercarriage", "stmt": "Undercarriage is a type of man-made object.", "wn_defs": ["framework that serves as a support for the body of a vehicle"]}, {"id": 216, "ent": "blacksmith", "stmt": "Blacksmith is a type of artist.", "wn_defs": ["a smith who forges and shapes iron with a hammer and anvil"]}, {"id": 217, "ent": "erica", "stmt": "Erica is a type of game.", "wn_defs": ["any plant of the genus Erica"]}, {"id": 218, "ent": "clinician", "stmt": "Clinician is a type of person.", "wn_defs": ["a practitioner (of medicine or psychology) who does clinical work instead of laboratory experiments"]}, {"id": 219, "ent": "beekeeper", "stmt": "Beekeeper is a type of person.", "wn_defs": ["a farmer who keeps bees for their honey"]}, {"id": 220, "ent": "purveyor", "stmt": "Purveyor is a type of living thing.", "wn_defs": ["someone who supplies provisions (especially food)"]}, {"id": 221, "ent": "carbine", "stmt": "Carbine is a type of man-made object.", "wn_defs": ["light automatic rifle"]}, {"id": 222, "ent": "ani", "stmt": "Ani is a type of animal.", "wn_defs": ["black tropical American cuckoo"]}, {"id": 223, "ent": "buyer", "stmt": "Buyer is a type of material.", "wn_defs": ["a person who buys"]}, {"id": 224, "ent": "bankrupt", "stmt": "Bankrupt is a type of man-made object.", "wn_defs": ["someone who has insufficient assets to cover their debts"]}, {"id": 225, "ent": "landlady", "stmt": "Landlady is a type of flower.", "wn_defs": ["a landlord who is a woman"]}, {"id": 226, "ent": "winger", "stmt": "Winger is a type of aircraft.", "wn_defs": ["(sports) player in wing position"]}, {"id": 227, "ent": "meatball", "stmt": "Meatball is a type of food.", "wn_defs": ["ground meat formed into a ball and fried or simmered in broth"]}, {"id": 228, "ent": "autofocus", "stmt": "Autofocus is a type of man-made object.", "wn_defs": ["an optical device for focussing a camera or other instrument automatically"]}, {"id": 229, "ent": "narrator", "stmt": "Narrator is a type of clothing.", "wn_defs": ["someone who tells a story"]}, {"id": 230, "ent": "winch", "stmt": "Winch is a type of musical instrument.", "wn_defs": ["lifting device consisting of a horizontal cylinder turned by a crank on which a cable or rope winds"]}, {"id": 231, "ent": "hoist", "stmt": "Hoist is a type of car.", "wn_defs": ["lifting device for raising heavy or cumbersome objects"]}, {"id": 232, "ent": "poodle", "stmt": "Poodle is a type of disease.", "wn_defs": ["an intelligent dog with a heavy curly solid-colored coat that is usually clipped; an old breed sometimes trained as sporting dogs or as performing dogs"]}, {"id": 233, "ent": "hamster", "stmt": "Hamster is a type of athlete.", "wn_defs": ["short-tailed Old World burrowing rodent with large cheek pouches"]}, {"id": 234, "ent": "finch", "stmt": "Finch is a type of bird.", "wn_defs": ["any of numerous small songbirds with short stout bills adapted for crushing seeds"]}, {"id": 235, "ent": "shrine", "stmt": "Shrine is a type of vehicle.", "wn_defs": ["a place of worship hallowed by association with some sacred thing or person"]}, {"id": 236, "ent": "ivy", "stmt": "Ivy is a type of car.", "wn_defs": ["Old World vine with lobed evergreen leaves and black berrylike fruits"]}, {"id": 237, "ent": "pensioner", "stmt": "Pensioner is a type of person.", "wn_defs": ["the beneficiary of a pension fund"]}, {"id": 238, "ent": "flatmate", "stmt": "Flatmate is a type of organ.", "wn_defs": ["an associate who shares an apartment with you"]}, {"id": 239, "ent": "communicator", "stmt": "Communicator is a type of living thing.", "wn_defs": ["a person who communicates with others"]}, {"id": 240, "ent": "menial", "stmt": "Menial is a type of plant.", "wn_defs": ["a domestic servant"]}, {"id": 241, "ent": "grandparent", "stmt": "Grandparent is a type of food.", "wn_defs": ["a parent of your father or mother"]}, {"id": 242, "ent": "chalet", "stmt": "Chalet is a type of building.", "wn_defs": ["a Swiss house with a sloping roof and wide eaves or a house built in this style"]}, {"id": 243, "ent": "avenger", "stmt": "Avenger is a type of vegetable.", "wn_defs": ["someone who takes vengeance"]}, {"id": 244, "ent": "gazebo", "stmt": "Gazebo is a type of fish.", "wn_defs": ["a small roofed building affording shade and rest"]}, {"id": 245, "ent": "marten", "stmt": "Marten is a type of living thing.", "wn_defs": ["agile slender-bodied arboreal mustelids somewhat larger than weasels"]}, {"id": 246, "ent": "uterus", "stmt": "Uterus is a type of container.", "wn_defs": ["a hollow muscular organ in the pelvic cavity of females; contains the developing fetus"]}, {"id": 247, "ent": "mockingbird", "stmt": "Mockingbird is a type of person.", "wn_defs": ["long-tailed grey-and-white songbird of the southern United States able to mimic songs of other birds"]}, {"id": 248, "ent": "milkman", "stmt": "Milkman is a type of living thing.", "wn_defs": ["someone who delivers milk"]}, {"id": 249, "ent": "sauna", "stmt": "Sauna is a type of animal.", "wn_defs": ["a Finnish steam bath; steam is produced by pouring water over heated rocks"]}, {"id": 250, "ent": "aviator", "stmt": "Aviator is a type of man-made object.", "wn_defs": ["someone who operates an aircraft"]}, {"id": 251, "ent": "freeman", "stmt": "Freeman is a type of worker.", "wn_defs": ["a person who is not a serf or a slave"]}, {"id": 252, "ent": "salesman", "stmt": "Salesman is a type of fish.", "wn_defs": ["a man salesperson"]}, {"id": 253, "ent": "synchrotron", "stmt": "Synchrotron is a type of device.", "wn_defs": ["cyclotron in which the electric field is maintained at a constant frequency"]}, {"id": 254, "ent": "traveler", "stmt": "Traveler is a type of reptile.", "wn_defs": ["a person who changes location"]}, {"id": 255, "ent": "skewer", "stmt": "Skewer is a type of game.", "wn_defs": ["a long pin for holding meat in position while it is being roasted"]}, {"id": 256, "ent": "sleuth", "stmt": "Sleuth is a type of metal.", "wn_defs": ["a detective who follows a trail"]}, {"id": 257, "ent": "abolitionist", "stmt": "Abolitionist is a type of artist.", "wn_defs": ["a reformer who favors abolishing slavery"]}, {"id": 258, "ent": "mantel", "stmt": "Mantel is a type of device.", "wn_defs": ["shelf that projects from wall above fireplace"]}, {"id": 259, "ent": "consulate", "stmt": "Consulate is a type of building.", "wn_defs": ["diplomatic building that serves as the residence or workplace of a consul"]}, {"id": 260, "ent": "minster", "stmt": "Minster is a type of boat.", "wn_defs": ["any of certain cathedrals and large churches; originally connected to a monastery"]}, {"id": 261, "ent": "informer", "stmt": "Informer is a type of fruit.", "wn_defs": ["one who reveals confidential information in return for money"]}, {"id": 262, "ent": "waitress", "stmt": "Waitress is a type of food.", "wn_defs": ["a woman waiter"]}, {"id": 263, "ent": "loudspeaker", "stmt": "Loudspeaker is a type of man-made object.", "wn_defs": ["electro-acoustic transducer that converts electrical signals into sounds loud enough to be heard at a distance"]}, {"id": 264, "ent": "orator", "stmt": "Orator is a type of footwear.", "wn_defs": ["a person who delivers a speech or oration"]}, {"id": 265, "ent": "drummer", "stmt": "Drummer is a type of worker.", "wn_defs": ["someone who plays a drum"]}, {"id": 266, "ent": "dingo", "stmt": "Dingo is a type of animal.", "wn_defs": ["wolflike yellowish-brown wild dog of Australia"]}, {"id": 267, "ent": "sailboat", "stmt": "Sailboat is a type of machine.", "wn_defs": ["a small sailing vessel; usually with a single mast"]}, {"id": 268, "ent": "idealist", "stmt": "Idealist is a type of disease.", "wn_defs": ["someone guided more by ideals than by practical considerations"]}, {"id": 269, "ent": "cactus", "stmt": "Cactus is a type of weapon.", "wn_defs": ["any succulent plant of the family Cactaceae native chiefly to arid regions of the New World and usually having spines"]}, {"id": 270, "ent": "basset", "stmt": "Basset is a type of animal.", "wn_defs": ["smooth-haired breed of hound with short legs and long ears"]}, {"id": 271, "ent": "handkerchief", "stmt": "Handkerchief is a type of man-made object.", "wn_defs": ["a square piece of cloth used for wiping the eyes or nose or as a costume accessory"]}, {"id": 272, "ent": "plumber", "stmt": "Plumber is a type of tree.", "wn_defs": ["a craftsman who installs and repairs pipes and fixtures and appliances"]}, {"id": 273, "ent": "webcam", "stmt": "Webcam is a type of man-made object.", "wn_defs": ["a digital camera designed to take digital photographs and transmit them over the internet"]}, {"id": 274, "ent": "dinghy", "stmt": "Dinghy is a type of boat.", "wn_defs": ["a small boat of shallow draft with cross thwarts for seats and rowlocks for oars with which it is propelled"]}, {"id": 275, "ent": "mod", "stmt": "Mod is a type of metal.", "wn_defs": ["a British teenager or young adult in the 1960s; noted for their clothes consciousness and opposition to the rockers"]}, {"id": 276, "ent": "erotic", "stmt": "Erotic is a type of flower.", "wn_defs": ["an erotic person"]}, {"id": 277, "ent": "songbird", "stmt": "Songbird is a type of room.", "wn_defs": ["any bird having a musical call"]}, {"id": 278, "ent": "sauerkraut", "stmt": "Sauerkraut is a type of fruit.", "wn_defs": ["shredded cabbage fermented in brine"]}, {"id": 279, "ent": "plenipotentiary", "stmt": "Plenipotentiary is a type of equipment.", "wn_defs": ["a diplomat who is fully authorized to represent his or her government"]}, {"id": 280, "ent": "dispensary", "stmt": "Dispensary is a type of man-made object.", "wn_defs": ["clinic where medicine and medical supplies are dispensed"]}, {"id": 281, "ent": "fetus", "stmt": "Fetus is a type of man-made object.", "wn_defs": ["an unborn or unhatched vertebrate in the later stages of development showing the main recognizable features of the mature animal"]}, {"id": 282, "ent": "orchid", "stmt": "Orchid is a type of artist.", "wn_defs": ["any of numerous plants of the orchid family usually having flowers of unusual shapes and beautiful colors"]}, {"id": 283, "ent": "basset", "stmt": "Basset is a type of dog.", "wn_defs": ["smooth-haired breed of hound with short legs and long ears"]}, {"id": 284, "ent": "coordinator", "stmt": "Coordinator is a type of person.", "wn_defs": ["someone whose task is to see that work goes harmoniously"]}, {"id": 285, "ent": "taster", "stmt": "Taster is a type of worker.", "wn_defs": ["someone who samples food or drink for its quality"]}, {"id": 286, "ent": "swot", "stmt": "Swot is a type of sport.", "wn_defs": ["an insignificant student who is ridiculed as being affected or boringly studious"]}, {"id": 287, "ent": "exporter", "stmt": "Exporter is a type of material.", "wn_defs": ["a businessperson who transports goods abroad (for sale)"]}, {"id": 288, "ent": "underwear", "stmt": "Underwear is a type of clothing.", "wn_defs": ["undergarment worn next to the skin and under the outer garments"]}, {"id": 289, "ent": "pendulum", "stmt": "Pendulum is a type of device.", "wn_defs": ["an apparatus consisting of an object mounted so that it swings freely under the influence of gravity"]}, {"id": 290, "ent": "stoner", "stmt": "Stoner is a type of bird.", "wn_defs": ["an attacker who pelts the victim with stones (especially with intent to kill)"]}, {"id": 291, "ent": "bartender", "stmt": "Bartender is a type of game.", "wn_defs": ["an employee who mixes and serves alcoholic drinks at a bar"]}, {"id": 292, "ent": "juggler", "stmt": "Juggler is a type of living thing.", "wn_defs": ["a performer who juggles objects and performs tricks of manual dexterity"]}, {"id": 293, "ent": "sovereign", "stmt": "Sovereign is a type of game.", "wn_defs": ["a nation's ruler or head of state usually by hereditary right"]}, {"id": 294, "ent": "stepson", "stmt": "Stepson is a type of mammal.", "wn_defs": ["the son your spouse by a former marriage"]}, {"id": 295, "ent": "mongoose", "stmt": "Mongoose is a type of living thing.", "wn_defs": ["agile grizzled Old World viverrine; preys on snakes and rodents"]}, {"id": 296, "ent": "hater", "stmt": "Hater is a type of dog.", "wn_defs": ["a person who hates"]}, {"id": 297, "ent": "cutthroat", "stmt": "Cutthroat is a type of game.", "wn_defs": ["someone who murders by cutting the victim's throat"]}, {"id": 298, "ent": "pedophile", "stmt": "Pedophile is a type of food.", "wn_defs": ["an adult who is sexually attracted to children"]}, {"id": 299, "ent": "ripper", "stmt": "Ripper is a type of plant.", "wn_defs": ["a murderer who slashes the victims with a knife"]}, {"id": 300, "ent": "sultan", "stmt": "Sultan is a type of organ.", "wn_defs": ["the ruler of a Muslim country (especially of the former Ottoman Empire)"]}, {"id": 301, "ent": "assistant", "stmt": "Assistant is a type of dish.", "wn_defs": ["a person who contributes to the fulfillment of a need or furtherance of an effort or purpose"]}, {"id": 302, "ent": "coauthor", "stmt": "Coauthor is a type of vegetable.", "wn_defs": ["a writer who collaborates with others in writing something"]}, {"id": 303, "ent": "ladybug", "stmt": "Ladybug is a type of equipment.", "wn_defs": ["small round bright-colored and spotted beetle that usually feeds on aphids and other insect pests"]}, {"id": 304, "ent": "dredge", "stmt": "Dredge is a type of man-made object.", "wn_defs": ["a power shovel to remove material from a channel or riverbed"]}, {"id": 305, "ent": "clarence", "stmt": "Clarence is a type of vehicle.", "wn_defs": ["a closed carriage with four wheels and seats for four passengers"]}, {"id": 306, "ent": "eft", "stmt": "Eft is a type of aircraft.", "wn_defs": ["a newt in its terrestrial stage of development"]}, {"id": 307, "ent": "modernist", "stmt": "Modernist is a type of building.", "wn_defs": ["an artist who makes a deliberate break with previous styles"]}, {"id": 308, "ent": "satchel", "stmt": "Satchel is a type of man-made object.", "wn_defs": ["luggage consisting of a small case with a flat bottom and (usually) a shoulder strap"]}, {"id": 309, "ent": "invalid", "stmt": "Invalid is a type of game.", "wn_defs": ["someone who is incapacitated by a chronic illness or injury"]}, {"id": 310, "ent": "jogger", "stmt": "Jogger is a type of food.", "wn_defs": ["someone who runs a steady slow pace (usually for exercise)"]}, {"id": 311, "ent": "rancher", "stmt": "Rancher is a type of body part.", "wn_defs": ["a person who owns or operates a ranch"]}, {"id": 312, "ent": "clary", "stmt": "Clary is a type of body part.", "wn_defs": ["aromatic herb of southern Europe; cultivated in Great Britain as a potherb and widely as an ornamental"]}, {"id": 313, "ent": "recliner", "stmt": "Recliner is a type of body part.", "wn_defs": ["an armchair whose back can be lowered and foot can be raised to allow the sitter to recline in it"]}, {"id": 314, "ent": "mercenary", "stmt": "Mercenary is a type of worker.", "wn_defs": ["a person hired to fight for another country than their own"]}, {"id": 315, "ent": "storeroom", "stmt": "Storeroom is a type of vegetable.", "wn_defs": ["a room in which things are stored"]}, {"id": 316, "ent": "protege", "stmt": "Protege is a type of vegetable.", "wn_defs": ["a person who receives support and protection from an influential patron who furthers the protege's career"]}, {"id": 317, "ent": "handyman", "stmt": "Handyman is a type of weapon.", "wn_defs": ["a man skilled in various odd jobs and other small tasks"]}, {"id": 318, "ent": "wanton", "stmt": "Wanton is a type of living thing.", "wn_defs": ["lewd or lascivious woman"]}, {"id": 319, "ent": "bloodhound", "stmt": "Bloodhound is a type of vegetable.", "wn_defs": ["a breed of large powerful hound of European origin having very acute smell and used in tracking"]}, {"id": 320, "ent": "crustacean", "stmt": "Crustacean is a type of body part.", "wn_defs": ["any mainly aquatic arthropod usually having a segmented body and chitinous exoskeleton"]}, {"id": 321, "ent": "electric", "stmt": "Electric is a type of car.", "wn_defs": ["a car that is powered by electricity"]}, {"id": 322, "ent": "itinerant", "stmt": "Itinerant is a type of artist.", "wn_defs": ["a laborer who moves from place to place as demanded by employment"]}, {"id": 323, "ent": "outhouse", "stmt": "Outhouse is a type of container.", "wn_defs": ["a small outbuilding with a bench having holes through which a user can defecate"]}, {"id": 324, "ent": "searchlight", "stmt": "Searchlight is a type of metal.", "wn_defs": ["a light source with reflectors that projects a beam of light in a particular direction"]}, {"id": 325, "ent": "stickler", "stmt": "Stickler is a type of person.", "wn_defs": ["someone who insists on something"]}, {"id": 326, "ent": "snob", "stmt": "Snob is a type of person.", "wn_defs": ["a person regarded as arrogant and annoying"]}, {"id": 327, "ent": "erotic", "stmt": "Erotic is a type of living thing.", "wn_defs": ["an erotic person"]}, {"id": 328, "ent": "violin", "stmt": "Violin is a type of musical instrument.", "wn_defs": ["bowed stringed instrument that is the highest member of the violin family; this instrument has four strings and a hollow body and an unfretted fingerboard and is played with a bow"]}, {"id": 329, "ent": "plover", "stmt": "Plover is a type of bird.", "wn_defs": ["any of numerous chiefly shorebirds of relatively compact build having straight bills and large pointed wings; closely related to the sandpipers"]}, {"id": 330, "ent": "lobbyist", "stmt": "Lobbyist is a type of boat.", "wn_defs": ["someone who is employed to persuade legislators to vote for legislation that favors the lobbyist's employer"]}, {"id": 331, "ent": "motorcyclist", "stmt": "Motorcyclist is a type of footwear.", "wn_defs": ["a traveler who rides a motorcycle"]}, {"id": 332, "ent": "harlequin", "stmt": "Harlequin is a type of reptile.", "wn_defs": ["a clown or buffoon (after the Harlequin character in the commedia dell'arte)"]}, {"id": 333, "ent": "cheeseburger", "stmt": "Cheeseburger is a type of man-made object.", "wn_defs": ["a hamburger with melted cheese on it"]}, {"id": 334, "ent": "wearer", "stmt": "Wearer is a type of mammal.", "wn_defs": ["a person who wears or carries or displays something as a body covering or accessory"]}, {"id": 335, "ent": "geisha", "stmt": "Geisha is a type of plant.", "wn_defs": ["a Japanese woman trained to entertain men with conversation and singing and dancing"]}, {"id": 336, "ent": "poppy", "stmt": "Poppy is a type of sport.", "wn_defs": ["annual or biennial or perennial herbs having showy flowers"]}, {"id": 337, "ent": "bobbin", "stmt": "Bobbin is a type of fruit.", "wn_defs": ["a winder around which thread or tape or film or other flexible materials can be wound"]}, {"id": 338, "ent": "preacher", "stmt": "Preacher is a type of person.", "wn_defs": ["someone whose occupation is preaching the gospel"]}, {"id": 339, "ent": "ladybug", "stmt": "Ladybug is a type of insect.", "wn_defs": ["small round bright-colored and spotted beetle that usually feeds on aphids and other insect pests"]}, {"id": 340, "ent": "cubby", "stmt": "Cubby is a type of living thing.", "wn_defs": ["a small secluded room"]}, {"id": 341, "ent": "neurotic", "stmt": "Neurotic is a type of metal.", "wn_defs": ["a person suffering from neurosis"]}, {"id": 342, "ent": "applicator", "stmt": "Applicator is a type of device.", "wn_defs": ["a device for applying a substance"]}, {"id": 343, "ent": "hoarder", "stmt": "Hoarder is a type of mammal.", "wn_defs": ["a person who accumulates things and hides them away for future use"]}, {"id": 344, "ent": "escapist", "stmt": "Escapist is a type of game.", "wn_defs": ["a person who escapes into a world of fantasy"]}, {"id": 345, "ent": "bereaved", "stmt": "Bereaved is a type of living thing.", "wn_defs": ["a person who has suffered the death of someone they loved"]}, {"id": 346, "ent": "termite", "stmt": "Termite is a type of mammal.", "wn_defs": ["whitish soft-bodied ant-like social insect that feeds on wood"]}, {"id": 347, "ent": "dragonfly", "stmt": "Dragonfly is a type of fish.", "wn_defs": ["slender-bodied non-stinging insect having iridescent wings that are outspread at rest; adults and nymphs feed on mosquitoes etc."]}, {"id": 348, "ent": "razor", "stmt": "Razor is a type of weapon.", "wn_defs": ["edge tool used in shaving"]}, {"id": 349, "ent": "mensch", "stmt": "Mensch is a type of person.", "wn_defs": ["a decent responsible person with admirable characteristics"]}, {"id": 350, "ent": "kingfisher", "stmt": "Kingfisher is a type of insect.", "wn_defs": ["nonpasserine large-headed bird with a short tail and long sharp bill; usually crested and bright-colored; feed mostly on fish"]}, {"id": 351, "ent": "snob", "stmt": "Snob is a type of vegetable.", "wn_defs": ["a person regarded as arrogant and annoying"]}, {"id": 352, "ent": "bock", "stmt": "Bock is a type of vegetable.", "wn_defs": ["a very strong lager traditionally brewed in the fall and aged through the winter for consumption in the spring"]}, {"id": 353, "ent": "accelerometer", "stmt": "Accelerometer is a type of sport.", "wn_defs": ["an instrument for measuring the acceleration of aircraft or rockets"]}, {"id": 354, "ent": "electrode", "stmt": "Electrode is a type of equipment.", "wn_defs": ["a conductor used to make electrical contact with some part of a circuit"]}, {"id": 355, "ent": "subcontractor", "stmt": "Subcontractor is a type of food.", "wn_defs": ["someone who enters into a subcontract with the primary contractor"]}, {"id": 356, "ent": "vase", "stmt": "Vase is a type of container.", "wn_defs": ["an open jar of glass or porcelain used as an ornament or to hold flowers"]}, {"id": 357, "ent": "parachute", "stmt": "Parachute is a type of equipment.", "wn_defs": ["rescue equipment consisting of a device that fills with air and retards your fall"]}, {"id": 358, "ent": "soloist", "stmt": "Soloist is a type of metal.", "wn_defs": ["a musician who performs a solo"]}, {"id": 359, "ent": "prankster", "stmt": "Prankster is a type of person.", "wn_defs": ["someone who plays practical jokes on others"]}, {"id": 360, "ent": "missus", "stmt": "Missus is a type of person.", "wn_defs": ["informal term of address for someone's wife"]}, {"id": 361, "ent": "gynecologist", "stmt": "Gynecologist is a type of living thing.", "wn_defs": ["a specialist in gynecology"]}, {"id": 362, "ent": "landowner", "stmt": "Landowner is a type of person.", "wn_defs": ["a holder or proprietor of land"]}, {"id": 363, "ent": "mosquito", "stmt": "Mosquito is a type of mammal.", "wn_defs": ["two-winged insect whose female has a long proboscis to pierce the skin and suck the blood of humans and animals"]}, {"id": 364, "ent": "decadent", "stmt": "Decadent is a type of dish.", "wn_defs": ["a person who has fallen into a decadent state (morally or artistically)"]}, {"id": 365, "ent": "toolbox", "stmt": "Toolbox is a type of tool.", "wn_defs": ["a box or chest or cabinet for holding hand tools"]}, {"id": 366, "ent": "brandy", "stmt": "Brandy is a type of food.", "wn_defs": ["distilled from wine or fermented fruit juice"]}, {"id": 367, "ent": "housekeeper", "stmt": "Housekeeper is a type of flower.", "wn_defs": ["a servant who is employed to perform domestic task in a household"]}, {"id": 368, "ent": "convener", "stmt": "Convener is a type of material.", "wn_defs": ["the member of a group whose duty it is to convene meetings"]}, {"id": 369, "ent": "moa", "stmt": "Moa is a type of animal.", "wn_defs": ["extinct flightless bird of New Zealand"]}, {"id": 370, "ent": "advertiser", "stmt": "Advertiser is a type of metal.", "wn_defs": ["someone whose business is advertising"]}, {"id": 371, "ent": "spaniel", "stmt": "Spaniel is a type of dog.", "wn_defs": ["any of several breeds of small to medium-sized gun dogs with a long silky coat and long frilled ears"]}, {"id": 372, "ent": "denture", "stmt": "Denture is a type of man-made object.", "wn_defs": ["a dental appliance that artificially replaces missing teeth"]}, {"id": 373, "ent": "pigeon", "stmt": "Pigeon is a type of person.", "wn_defs": ["wild and domesticated birds having a heavy body and short legs"]}, {"id": 374, "ent": "chrysalis", "stmt": "Chrysalis is a type of body part.", "wn_defs": ["pupa of a moth or butterfly enclosed in a cocoon"]}, {"id": 375, "ent": "vizier", "stmt": "Vizier is a type of athlete.", "wn_defs": ["a high official in a Muslim government (especially in the Ottoman Empire)"]}, {"id": 376, "ent": "bandit", "stmt": "Bandit is a type of device.", "wn_defs": ["an armed thief who is (usually) a member of a band"]}, {"id": 377, "ent": "dominatrix", "stmt": "Dominatrix is a type of living thing.", "wn_defs": ["a dominating woman (especially one who plays that role in a sadomasochistic sexual relationship)"]}, {"id": 378, "ent": "protectionist", "stmt": "Protectionist is a type of plant.", "wn_defs": ["an advocate of protectionism"]}, {"id": 379, "ent": "cyborg", "stmt": "Cyborg is a type of body part.", "wn_defs": ["a human being whose body has been taken over in whole or in part by electromechanical devices"]}, {"id": 380, "ent": "manger", "stmt": "Manger is a type of worker.", "wn_defs": ["a container (usually in a barn or stable) from which cattle or horses feed"]}, {"id": 381, "ent": "stowaway", "stmt": "Stowaway is a type of vegetable.", "wn_defs": ["a person who hides aboard a ship or plane in the hope of getting free passage"]}, {"id": 382, "ent": "stingray", "stmt": "Stingray is a type of living thing.", "wn_defs": ["large venomous ray with large barbed spines near the base of a thin whiplike tail capable of inflicting severe wounds"]}, {"id": 383, "ent": "immigrant", "stmt": "Immigrant is a type of game.", "wn_defs": ["a person who comes to a country where they were not born in order to settle there"]}, {"id": 384, "ent": "peripheral", "stmt": "Peripheral is a type of animal.", "wn_defs": ["(computer science) electronic equipment connected by cable to the CPU of a computer"]}, {"id": 385, "ent": "corgi", "stmt": "Corgi is a type of clothing.", "wn_defs": ["either of two Welsh breeds of long-bodied short-legged dogs with erect ears and a fox-like head"]}, {"id": 386, "ent": "trusty", "stmt": "Trusty is a type of game.", "wn_defs": ["a convict who is considered trustworthy and granted special privileges"]}, {"id": 387, "ent": "inventor", "stmt": "Inventor is a type of drink.", "wn_defs": ["someone who is the first to think of or make something"]}, {"id": 388, "ent": "narc", "stmt": "Narc is a type of person.", "wn_defs": ["a lawman concerned with narcotics violations"]}, {"id": 389, "ent": "brazier", "stmt": "Brazier is a type of drink.", "wn_defs": ["large metal container in which coal or charcoal is burned; warms people who must stay outside for long times"]}, {"id": 390, "ent": "curate", "stmt": "Curate is a type of living thing.", "wn_defs": ["a person authorized to conduct religious worship"]}, {"id": 391, "ent": "stagecoach", "stmt": "Stagecoach is a type of vehicle.", "wn_defs": ["a large coach-and-four formerly used to carry passengers and mail on regular routes between towns"]}, {"id": 392, "ent": "molester", "stmt": "Molester is a type of mammal.", "wn_defs": ["someone who subjects others to unwanted or improper sexual activities"]}, {"id": 393, "ent": "paramedic", "stmt": "Paramedic is a type of worker.", "wn_defs": ["a person trained to assist medical professionals and to give emergency medical treatment"]}, {"id": 394, "ent": "inventor", "stmt": "Inventor is a type of body part.", "wn_defs": ["someone who is the first to think of or make something"]}, {"id": 395, "ent": "synagogue", "stmt": "Synagogue is a type of vegetable.", "wn_defs": ["(Judaism) the place of worship for a Jewish congregation"]}, {"id": 396, "ent": "adolescent", "stmt": "Adolescent is a type of mammal.", "wn_defs": ["a juvenile between the onset of puberty and maturity"]}, {"id": 397, "ent": "contrarian", "stmt": "Contrarian is a type of metal.", "wn_defs": ["an investor who deliberately decides to go against the prevailing wisdom of other investors"]}, {"id": 398, "ent": "swimsuit", "stmt": "Swimsuit is a type of equipment.", "wn_defs": ["tight fitting garment worn for swimming"]}, {"id": 399, "ent": "shrine", "stmt": "Shrine is a type of container.", "wn_defs": ["a place of worship hallowed by association with some sacred thing or person"]}, {"id": 400, "ent": "manatee", "stmt": "Manatee is a type of vehicle.", "wn_defs": ["sirenian mammal of tropical coastal waters of America; the flat tail is rounded"]}, {"id": 401, "ent": "toddler", "stmt": "Toddler is a type of tool.", "wn_defs": ["a young child"]}, {"id": 402, "ent": "introvert", "stmt": "Introvert is a type of worker.", "wn_defs": ["(psychology) a person who tends to shrink from social contacts and to become preoccupied with their own thoughts"]}, {"id": 403, "ent": "copywriter", "stmt": "Copywriter is a type of clothing.", "wn_defs": ["a person employed to write advertising or publicity copy"]}, {"id": 404, "ent": "vigilante", "stmt": "Vigilante is a type of person.", "wn_defs": ["member of a vigilance committee"]}, {"id": 405, "ent": "cuttlefish", "stmt": "Cuttlefish is a type of animal.", "wn_defs": ["ten-armed oval-bodied cephalopod with narrow fins as long as the body and a large calcareous internal shell"]}, {"id": 406, "ent": "alderman", "stmt": "Alderman is a type of athlete.", "wn_defs": ["a member of a municipal legislative body (as a city council)"]}, {"id": 407, "ent": "hater", "stmt": "Hater is a type of boat.", "wn_defs": ["a person who hates"]}, {"id": 408, "ent": "airship", "stmt": "Airship is a type of dog.", "wn_defs": ["a steerable self-propelled aircraft"]}, {"id": 409, "ent": "benefactor", "stmt": "Benefactor is a type of person.", "wn_defs": ["a person who helps people or institutions (especially with financial help)"]}, {"id": 410, "ent": "financier", "stmt": "Financier is a type of worker.", "wn_defs": ["a person skilled in large scale financial transactions"]}, {"id": 411, "ent": "trader", "stmt": "Trader is a type of worker.", "wn_defs": ["someone who purchases and maintains an inventory of goods to be sold"]}, {"id": 412, "ent": "anti", "stmt": "Anti is a type of living thing.", "wn_defs": ["a person who is opposed (to an action or policy or practice etc.)"]}, {"id": 413, "ent": "sarcophagus", "stmt": "Sarcophagus is a type of tool.", "wn_defs": ["a stone coffin (usually bearing sculpture or inscriptions)"]}, {"id": 414, "ent": "paleontologist", "stmt": "Paleontologist is a type of worker.", "wn_defs": ["a specialist in paleontology"]}, {"id": 415, "ent": "accordion", "stmt": "Accordion is a type of fish.", "wn_defs": ["a portable box-shaped free-reed instrument; the reeds are made to vibrate by air from the bellows controlled by the player"]}, {"id": 416, "ent": "sphincter", "stmt": "Sphincter is a type of disease.", "wn_defs": ["a ring of muscle that contracts to close an opening"]}, {"id": 417, "ent": "detainee", "stmt": "Detainee is a type of living thing.", "wn_defs": ["some held in custody"]}, {"id": 418, "ent": "pleaser", "stmt": "Pleaser is a type of game.", "wn_defs": ["a pleasing entertainer"]}, {"id": 419, "ent": "princess", "stmt": "Princess is a type of person.", "wn_defs": ["a female member of a royal family other than the queen (especially the daughter of a sovereign)"]}, {"id": 420, "ent": "paramedic", "stmt": "Paramedic is a type of dish.", "wn_defs": ["a person trained to assist medical professionals and to give emergency medical treatment"]}, {"id": 421, "ent": "rink", "stmt": "Rink is a type of sport.", "wn_defs": ["building that contains a surface for ice skating or roller skating"]}, {"id": 422, "ent": "propagandist", "stmt": "Propagandist is a type of person.", "wn_defs": ["a person who disseminates messages calculated to assist some cause or some government"]}, {"id": 423, "ent": "confidante", "stmt": "Confidante is a type of person.", "wn_defs": ["a female confidant"]}, {"id": 424, "ent": "stockbroker", "stmt": "Stockbroker is a type of footwear.", "wn_defs": ["an agent in the buying and selling of stocks and bonds"]}, {"id": 425, "ent": "drosophila", "stmt": "Drosophila is a type of food.", "wn_defs": ["small fruit fly used by Thomas Hunt Morgan in studying basic mechanisms of inheritance"]}, {"id": 426, "ent": "pacifist", "stmt": "Pacifist is a type of insect.", "wn_defs": ["someone opposed to violence as a means of settling disputes"]}, {"id": 427, "ent": "suitor", "stmt": "Suitor is a type of game.", "wn_defs": ["a man who courts a woman"]}, {"id": 428, "ent": "apologist", "stmt": "Apologist is a type of worker.", "wn_defs": ["a person who argues to defend or justify some policy or institution"]}, {"id": 429, "ent": "clubhouse", "stmt": "Clubhouse is a type of room.", "wn_defs": ["a building that is occupied by a social club"]}, {"id": 430, "ent": "strategist", "stmt": "Strategist is a type of person.", "wn_defs": ["an expert in strategy (especially in warfare)"]}, {"id": 431, "ent": "jeep", "stmt": "Jeep is a type of container.", "wn_defs": ["a car suitable for traveling over rough terrain"]}, {"id": 432, "ent": "survivalist", "stmt": "Survivalist is a type of fish.", "wn_defs": ["someone who tries to insure their personal survival or the survival of their group or nation"]}, {"id": 433, "ent": "vermouth", "stmt": "Vermouth is a type of plant.", "wn_defs": ["any of several white wines flavored with aromatic herbs; used as aperitifs or in mixed drinks"]}, {"id": 434, "ent": "contrarian", "stmt": "Contrarian is a type of animal.", "wn_defs": ["an investor who deliberately decides to go against the prevailing wisdom of other investors"]}, {"id": 435, "ent": "cattle", "stmt": "Cattle is a type of animal.", "wn_defs": ["domesticated bovine animals as a group regardless of sex or age; ; ; - Bible"]}, {"id": 436, "ent": "pensioner", "stmt": "Pensioner is a type of sport.", "wn_defs": ["the beneficiary of a pension fund"]}, {"id": 437, "ent": "marlin", "stmt": "Marlin is a type of living thing.", "wn_defs": ["large long-jawed oceanic sport fishes; related to sailfishes and spearfishes; not completely cold-blooded i.e. able to warm their brains and eyes"]}, {"id": 438, "ent": "mockingbird", "stmt": "Mockingbird is a type of bird.", "wn_defs": ["long-tailed grey-and-white songbird of the southern United States able to mimic songs of other birds"]}, {"id": 439, "ent": "dandelion", "stmt": "Dandelion is a type of mammal.", "wn_defs": ["any of several herbs of the genus Taraxacum having long tap roots and deeply notched leaves and bright yellow flowers followed by fluffy seed balls"]}, {"id": 440, "ent": "meatball", "stmt": "Meatball is a type of game.", "wn_defs": ["ground meat formed into a ball and fried or simmered in broth"]}, {"id": 441, "ent": "chime", "stmt": "Chime is a type of organ.", "wn_defs": ["a percussion instrument consisting of a set of tuned bells that are struck with a hammer; used as an orchestral instrument"]}, {"id": 442, "ent": "hornet", "stmt": "Hornet is a type of worker.", "wn_defs": ["large stinging paper wasp"]}, {"id": 443, "ent": "wildflower", "stmt": "Wildflower is a type of living thing.", "wn_defs": ["wild or uncultivated flowering plant"]}, {"id": 444, "ent": "gyroscope", "stmt": "Gyroscope is a type of machine.", "wn_defs": ["rotating mechanism in the form of a universally mounted spinning wheel that offers resistance to turns in any direction"]}, {"id": 445, "ent": "lender", "stmt": "Lender is a type of bird.", "wn_defs": ["someone who lends money or gives credit in business matters"]}, {"id": 446, "ent": "kangaroo", "stmt": "Kangaroo is a type of athlete.", "wn_defs": ["any of several herbivorous leaping marsupials of Australia and New Guinea having large powerful hind legs and a long thick tail"]}, {"id": 447, "ent": "calico", "stmt": "Calico is a type of fabric.", "wn_defs": ["coarse cloth with a bright print"]}, {"id": 448, "ent": "tongs", "stmt": "Tongs is a type of tool.", "wn_defs": ["any of various devices for taking hold of objects; usually have two hinged legs with handles above and pointed hooks below"]}, {"id": 449, "ent": "cadet", "stmt": "Cadet is a type of dish.", "wn_defs": ["a military trainee (as at a military academy)"]}, {"id": 450, "ent": "repairman", "stmt": "Repairman is a type of fruit.", "wn_defs": ["a skilled worker whose job is to repair things"]}, {"id": 451, "ent": "clubhouse", "stmt": "Clubhouse is a type of machine.", "wn_defs": ["a building that is occupied by a social club"]}, {"id": 452, "ent": "hummingbird", "stmt": "Hummingbird is a type of animal.", "wn_defs": ["tiny American bird having brilliant iridescent plumage and long slender bills; wings are specialized for vibrating flight"]}, {"id": 453, "ent": "bison", "stmt": "Bison is a type of mammal.", "wn_defs": ["any of several large humped bovids having shaggy manes and large heads and short horns"]}, {"id": 454, "ent": "piglet", "stmt": "Piglet is a type of mammal.", "wn_defs": ["a young pig"]}, {"id": 455, "ent": "hyena", "stmt": "Hyena is a type of plant.", "wn_defs": ["doglike nocturnal mammal of Africa and southern Asia that feeds chiefly on carrion"]}, {"id": 456, "ent": "wife", "stmt": "Wife is a type of mammal.", "wn_defs": ["a married woman; a man's partner in marriage"]}, {"id": 457, "ent": "berk", "stmt": "Berk is a type of living thing.", "wn_defs": ["a stupid person who is easy to take advantage of"]}, {"id": 458, "ent": "transistor", "stmt": "Transistor is a type of fish.", "wn_defs": ["a semiconductor device capable of amplification"]}, {"id": 459, "ent": "firstborn", "stmt": "Firstborn is a type of sport.", "wn_defs": ["the offspring who came first in the order of birth"]}, {"id": 460, "ent": "interviewee", "stmt": "Interviewee is a type of worker.", "wn_defs": ["a person who is interviewed"]}, {"id": 461, "ent": "surrealist", "stmt": "Surrealist is a type of fish.", "wn_defs": ["an artist who is a member of the movement called surrealism"]}, {"id": 462, "ent": "corrective", "stmt": "Corrective is a type of man-made object.", "wn_defs": ["a device for treating injury or disease"]}, {"id": 463, "ent": "searchlight", "stmt": "Searchlight is a type of equipment.", "wn_defs": ["a light source with reflectors that projects a beam of light in a particular direction"]}, {"id": 464, "ent": "bookkeeper", "stmt": "Bookkeeper is a type of reptile.", "wn_defs": ["someone who records the transactions of a business"]}, {"id": 465, "ent": "takin", "stmt": "Takin is a type of fish.", "wn_defs": ["large heavily built goat antelope of eastern Himalayan area"]}, {"id": 466, "ent": "therapist", "stmt": "Therapist is a type of living thing.", "wn_defs": ["a person skilled in a particular type of therapy"]}, {"id": 467, "ent": "communicator", "stmt": "Communicator is a type of person.", "wn_defs": ["a person who communicates with others"]}, {"id": 468, "ent": "bereaved", "stmt": "Bereaved is a type of game.", "wn_defs": ["a person who has suffered the death of someone they loved"]}, {"id": 469, "ent": "apparel", "stmt": "Apparel is a type of weapon.", "wn_defs": ["clothing in general"]}, {"id": 470, "ent": "boardroom", "stmt": "Boardroom is a type of room.", "wn_defs": ["a room where a committee meets (such as the board of directors of a company)"]}, {"id": 471, "ent": "goldilocks", "stmt": "Goldilocks is a type of flower.", "wn_defs": ["early-flowering perennial of southern and southeastern Europe with flower heads resembling those of goldenrod"]}, {"id": 472, "ent": "manatee", "stmt": "Manatee is a type of food.", "wn_defs": ["sirenian mammal of tropical coastal waters of America; the flat tail is rounded"]}, {"id": 473, "ent": "aloe", "stmt": "Aloe is a type of room.", "wn_defs": ["succulent plants having rosettes of leaves usually with fiber like hemp and spikes of showy flowers; found chiefly in Africa"]}, {"id": 474, "ent": "congressman", "stmt": "Congressman is a type of flower.", "wn_defs": ["a member of the United States House of Representatives"]}, {"id": 475, "ent": "ecologist", "stmt": "Ecologist is a type of worker.", "wn_defs": ["a biologist who studies the relation between organisms and their environment"]}, {"id": 476, "ent": "dahlia", "stmt": "Dahlia is a type of plant.", "wn_defs": ["any of several plants of or developed from the species Dahlia pinnata having tuberous roots and showy rayed variously colored flower heads; native to the mountains of Mexico and Central America and Colombia"]}, {"id": 477, "ent": "lodger", "stmt": "Lodger is a type of plant.", "wn_defs": ["a tenant in someone's house"]}, {"id": 478, "ent": "flea", "stmt": "Flea is a type of animal.", "wn_defs": ["any wingless bloodsucking parasitic insect noted for ability to leap"]}, {"id": 479, "ent": "hillbilly", "stmt": "Hillbilly is a type of living thing.", "wn_defs": ["a disparaging term for an unsophisticated person"]}, {"id": 480, "ent": "stewardess", "stmt": "Stewardess is a type of equipment.", "wn_defs": ["a woman steward on an airplane"]}, {"id": 481, "ent": "helmsman", "stmt": "Helmsman is a type of athlete.", "wn_defs": ["the person who steers a ship"]}, {"id": 482, "ent": "jasmine", "stmt": "Jasmine is a type of vegetable.", "wn_defs": ["any of several shrubs and vines of the genus Jasminum chiefly native to Asia"]}, {"id": 483, "ent": "palmetto", "stmt": "Palmetto is a type of musical instrument.", "wn_defs": ["any of several low-growing palms with fan-shaped leaves"]}, {"id": 484, "ent": "rapporteur", "stmt": "Rapporteur is a type of organ.", "wn_defs": ["a recorder appointed by a committee to prepare reports of the meetings"]}, {"id": 485, "ent": "bobcat", "stmt": "Bobcat is a type of mammal.", "wn_defs": ["small lynx of North America"]}, {"id": 486, "ent": "cutler", "stmt": "Cutler is a type of living thing.", "wn_defs": ["a dealer in cutlery"]}, {"id": 487, "ent": "hearse", "stmt": "Hearse is a type of man-made object.", "wn_defs": ["a vehicle for carrying a coffin to a church or a cemetery; formerly drawn by horses but now usually a motor vehicle"]}, {"id": 488, "ent": "spastic", "stmt": "Spastic is a type of reptile.", "wn_defs": ["a person suffering from spastic paralysis"]}, {"id": 489, "ent": "alcoholic", "stmt": "Alcoholic is a type of drink.", "wn_defs": ["a person who drinks alcohol to excess habitually"]}, {"id": 490, "ent": "longhorn", "stmt": "Longhorn is a type of tree.", "wn_defs": ["long-horned beef cattle formerly common in southwestern United States"]}, {"id": 491, "ent": "beagle", "stmt": "Beagle is a type of sport.", "wn_defs": ["a small short-legged smooth-coated breed of hound"]}, {"id": 492, "ent": "impressionist", "stmt": "Impressionist is a type of fabric.", "wn_defs": ["a painter who follows the theories of Impressionism"]}, {"id": 493, "ent": "inpatient", "stmt": "Inpatient is a type of fruit.", "wn_defs": ["a patient who is residing in the hospital where he is being treated"]}, {"id": 494, "ent": "emir", "stmt": "Emir is a type of boat.", "wn_defs": ["an independent ruler or chieftain (especially in Africa or Arabia)"]}, {"id": 495, "ent": "snapdragon", "stmt": "Snapdragon is a type of living thing.", "wn_defs": ["a garden plant of the genus Antirrhinum having showy white or yellow or crimson flowers resembling the face of a dragon"]}, {"id": 496, "ent": "stockholder", "stmt": "Stockholder is a type of furniture.", "wn_defs": ["someone who holds shares of stock in a corporation"]}, {"id": 497, "ent": "schizophrenic", "stmt": "Schizophrenic is a type of sport.", "wn_defs": ["someone who is afflicted with schizophrenia"]}, {"id": 498, "ent": "rapporteur", "stmt": "Rapporteur is a type of person.", "wn_defs": ["a recorder appointed by a committee to prepare reports of the meetings"]}, {"id": 499, "ent": "ombudsman", "stmt": "Ombudsman is a type of living thing.", "wn_defs": ["a government appointee who investigates complaints by private persons against the government"]}, {"id": 500, "ent": "cleric", "stmt": "Cleric is a type of plant.", "wn_defs": ["a clergyman or other person in religious orders"]}, {"id": 501, "ent": "patriot", "stmt": "Patriot is a type of aircraft.", "wn_defs": ["one who loves and defends his or her country"]}, {"id": 502, "ent": "homophobe", "stmt": "Homophobe is a type of sport.", "wn_defs": ["a person who hates or fears homosexual people"]}, {"id": 503, "ent": "rifle", "stmt": "Rifle is a type of animal.", "wn_defs": ["a shoulder firearm with a long barrel and a rifled bore"]}, {"id": 504, "ent": "phytoplankton", "stmt": "Phytoplankton is a type of plant.", "wn_defs": ["photosynthetic or plant constituent of plankton; mainly unicellular algae"]}, {"id": 505, "ent": "caveman", "stmt": "Caveman is a type of person.", "wn_defs": ["someone who lives in a cave"]}, {"id": 506, "ent": "exporter", "stmt": "Exporter is a type of drink.", "wn_defs": ["a businessperson who transports goods abroad (for sale)"]}, {"id": 507, "ent": "sibling", "stmt": "Sibling is a type of mammal.", "wn_defs": ["a person's brother or sister"]}, {"id": 508, "ent": "limousine", "stmt": "Limousine is a type of clothing.", "wn_defs": ["large luxurious car; usually driven by a chauffeur"]}, {"id": 509, "ent": "lyre", "stmt": "Lyre is a type of man-made object.", "wn_defs": ["a harp used by ancient Greeks for accompaniment"]}, {"id": 510, "ent": "audiovisual", "stmt": "Audiovisual is a type of material.", "wn_defs": ["materials using sight or sound to present information"]}, {"id": 511, "ent": "communicator", "stmt": "Communicator is a type of sport.", "wn_defs": ["a person who communicates with others"]}, {"id": 512, "ent": "bathrobe", "stmt": "Bathrobe is a type of metal.", "wn_defs": ["a loose-fitting robe of towelling; worn after a bath or swim"]}, {"id": 513, "ent": "thwart", "stmt": "Thwart is a type of device.", "wn_defs": ["a crosspiece spreading the gunnels of a boat; used as a seat in a rowboat"]}, {"id": 514, "ent": "stork", "stmt": "Stork is a type of aircraft.", "wn_defs": ["large mostly Old World wading birds typically having white-and-black plumage"]}, {"id": 515, "ent": "marksman", "stmt": "Marksman is a type of game.", "wn_defs": ["someone skilled in shooting"]}, {"id": 516, "ent": "confessional", "stmt": "Confessional is a type of toy.", "wn_defs": ["a booth where a priest sits to hear confessions"]}, {"id": 517, "ent": "monastery", "stmt": "Monastery is a type of container.", "wn_defs": ["the residence of a religious community"]}, {"id": 518, "ent": "headdress", "stmt": "Headdress is a type of flower.", "wn_defs": ["clothing for the head"]}, {"id": 519, "ent": "statesman", "stmt": "Statesman is a type of living thing.", "wn_defs": ["a man who is a respected leader in national or international affairs"]}, {"id": 520, "ent": "milkman", "stmt": "Milkman is a type of vegetable.", "wn_defs": ["someone who delivers milk"]}, {"id": 521, "ent": "thermometer", "stmt": "Thermometer is a type of tool.", "wn_defs": ["measuring instrument for measuring temperature"]}, {"id": 522, "ent": "expressionist", "stmt": "Expressionist is a type of game.", "wn_defs": ["an artist who is an adherent of expressionism"]}, {"id": 523, "ent": "perpetrator", "stmt": "Perpetrator is a type of insect.", "wn_defs": ["someone who perpetrates wrongdoing"]}, {"id": 524, "ent": "coupe", "stmt": "Coupe is a type of container.", "wn_defs": ["a car with two doors and front seats and a luggage compartment"]}, {"id": 525, "ent": "councilman", "stmt": "Councilman is a type of worker.", "wn_defs": ["a man who is a council member"]}, {"id": 526, "ent": "authoritarian", "stmt": "Authoritarian is a type of sport.", "wn_defs": ["a person who behaves in a tyrannical manner"]}, {"id": 527, "ent": "satirist", "stmt": "Satirist is a type of artist.", "wn_defs": ["a humorist who uses ridicule and irony and sarcasm"]}, {"id": 528, "ent": "antelope", "stmt": "Antelope is a type of living thing.", "wn_defs": ["graceful Old World ruminant with long legs and horns directed upward and backward; includes gazelles; springboks; impalas; addax; gerenuks; blackbucks; dik-diks"]}, {"id": 529, "ent": "scraper", "stmt": "Scraper is a type of device.", "wn_defs": ["any of various hand tools for scraping"]}, {"id": 530, "ent": "ballroom", "stmt": "Ballroom is a type of sport.", "wn_defs": ["large room used mainly for dancing"]}, {"id": 531, "ent": "dunce", "stmt": "Dunce is a type of flower.", "wn_defs": ["a stupid person; these words are used to express a low opinion of someone's intelligence"]}, {"id": 532, "ent": "hiker", "stmt": "Hiker is a type of athlete.", "wn_defs": ["a foot traveler; someone who goes on an extended walk (for pleasure)"]}, {"id": 533, "ent": "playmate", "stmt": "Playmate is a type of mammal.", "wn_defs": ["a companion at play"]}, {"id": 534, "ent": "agitator", "stmt": "Agitator is a type of vegetable.", "wn_defs": ["one who agitates; a political troublemaker"]}, {"id": 535, "ent": "stiletto", "stmt": "Stiletto is a type of footwear.", "wn_defs": ["a small dagger with a tapered blade"]}, {"id": 536, "ent": "clock", "stmt": "Clock is a type of machine.", "wn_defs": ["a timepiece that shows the time of day"]}, {"id": 537, "ent": "bankrupt", "stmt": "Bankrupt is a type of person.", "wn_defs": ["someone who has insufficient assets to cover their debts"]}, {"id": 538, "ent": "sophomore", "stmt": "Sophomore is a type of boat.", "wn_defs": ["a second-year undergraduate"]}, {"id": 539, "ent": "hillbilly", "stmt": "Hillbilly is a type of material.", "wn_defs": ["a disparaging term for an unsophisticated person"]}, {"id": 540, "ent": "yarrow", "stmt": "Yarrow is a type of flower.", "wn_defs": ["ubiquitous strong-scented mat-forming Eurasian herb of wasteland, hedgerow or pasture having narrow serrate leaves and small usually white florets; widely naturalized in North America"]}, {"id": 541, "ent": "caveman", "stmt": "Caveman is a type of living thing.", "wn_defs": ["someone who lives in a cave"]}, {"id": 542, "ent": "battleship", "stmt": "Battleship is a type of person.", "wn_defs": ["large and heavily armoured warship"]}, {"id": 543, "ent": "ratchet", "stmt": "Ratchet is a type of tool.", "wn_defs": ["mechanical device consisting of a toothed wheel or rack engaged with a pawl that permits it to move in only one direction"]}, {"id": 544, "ent": "rival", "stmt": "Rival is a type of weapon.", "wn_defs": ["the contestant you hope to defeat"]}, {"id": 545, "ent": "candlestick", "stmt": "Candlestick is a type of device.", "wn_defs": ["a holder with sockets for candles"]}, {"id": 546, "ent": "helmsman", "stmt": "Helmsman is a type of boat.", "wn_defs": ["the person who steers a ship"]}, {"id": 547, "ent": "suitor", "stmt": "Suitor is a type of person.", "wn_defs": ["a man who courts a woman"]}, {"id": 548, "ent": "greenhouse", "stmt": "Greenhouse is a type of device.", "wn_defs": ["a building with glass walls and roof; for the cultivation and exhibition of plants under controlled conditions"]}, {"id": 549, "ent": "socialist", "stmt": "Socialist is a type of person.", "wn_defs": ["a political advocate of socialism"]}, {"id": 550, "ent": "prince", "stmt": "Prince is a type of person.", "wn_defs": ["a male member of a royal family other than the sovereign (especially the son of a sovereign)"]}, {"id": 551, "ent": "pulley", "stmt": "Pulley is a type of footwear.", "wn_defs": ["a simple machine consisting of a wheel with a groove in which a rope can run to change the direction or point of application of a force applied to the rope"]}, {"id": 552, "ent": "parachute", "stmt": "Parachute is a type of device.", "wn_defs": ["rescue equipment consisting of a device that fills with air and retards your fall"]}, {"id": 553, "ent": "crocodile", "stmt": "Crocodile is a type of body part.", "wn_defs": ["large voracious aquatic reptile having a long snout with massive jaws and sharp teeth and a body covered with bony plates; of sluggish tropical waters"]}, {"id": 554, "ent": "innocent", "stmt": "Innocent is a type of man-made object.", "wn_defs": ["a person who lacks knowledge of evil"]}, {"id": 555, "ent": "conch", "stmt": "Conch is a type of living thing.", "wn_defs": ["any of various edible tropical marine gastropods of the genus Strombus having a brightly-colored spiral shell with large outer lip"]}, {"id": 556, "ent": "electrode", "stmt": "Electrode is a type of car.", "wn_defs": ["a conductor used to make electrical contact with some part of a circuit"]}, {"id": 557, "ent": "raglan", "stmt": "Raglan is a type of furniture.", "wn_defs": ["a garment (coat or sweater) that has raglan sleeves"]}, {"id": 558, "ent": "nacho", "stmt": "Nacho is a type of material.", "wn_defs": ["a tortilla chip topped with cheese and chili-pepper and broiled"]}, {"id": 559, "ent": "surgeon", "stmt": "Surgeon is a type of worker.", "wn_defs": ["a physician who specializes in surgery"]}, {"id": 560, "ent": "artist", "stmt": "Artist is a type of fish.", "wn_defs": ["a person whose creative work shows sensitivity and imagination"]}, {"id": 561, "ent": "claimant", "stmt": "Claimant is a type of disease.", "wn_defs": ["someone who claims a benefit or right or title"]}, {"id": 562, "ent": "ibis", "stmt": "Ibis is a type of footwear.", "wn_defs": ["wading birds of warm regions having long slender down-curved bills"]}, {"id": 563, "ent": "notary", "stmt": "Notary is a type of person.", "wn_defs": ["someone legally empowered to witness signatures and certify a document's validity and to take depositions"]}, {"id": 564, "ent": "motorboat", "stmt": "Motorboat is a type of machine.", "wn_defs": ["a boat propelled by an internal-combustion engine"]}, {"id": 565, "ent": "loom", "stmt": "Loom is a type of man-made object.", "wn_defs": ["a textile machine for weaving yarn into a textile"]}, {"id": 566, "ent": "firearm", "stmt": "Firearm is a type of weapon.", "wn_defs": ["a portable gun"]}, {"id": 567, "ent": "haggis", "stmt": "Haggis is a type of organ.", "wn_defs": ["made of sheep's or calf's viscera minced with oatmeal and suet and onions and boiled in the animal's stomach"]}, {"id": 568, "ent": "brogan", "stmt": "Brogan is a type of food.", "wn_defs": ["a thick and heavy shoe"]}, {"id": 569, "ent": "tam", "stmt": "Tam is a type of dish.", "wn_defs": ["a woolen cap of Scottish origin"]}, {"id": 570, "ent": "motorcyclist", "stmt": "Motorcyclist is a type of sport.", "wn_defs": ["a traveler who rides a motorcycle"]}, {"id": 571, "ent": "alumnus", "stmt": "Alumnus is a type of plant.", "wn_defs": ["a person who has received a degree from a school (high school or college or university)"]}, {"id": 572, "ent": "dildo", "stmt": "Dildo is a type of equipment.", "wn_defs": ["a vibrating device that substitutes for an erect penis to provide vaginal stimulation"]}, {"id": 573, "ent": "tights", "stmt": "Tights is a type of vegetable.", "wn_defs": ["skintight knit hose covering the body from the waist to the feet worn by acrobats and dancers and as stockings by women and girls"]}, {"id": 574, "ent": "doorbell", "stmt": "Doorbell is a type of boat.", "wn_defs": ["a push button at an outer door that gives a ringing or buzzing signal when pushed"]}, {"id": 575, "ent": "letterman", "stmt": "Letterman is a type of reptile.", "wn_defs": ["an athlete who has earned a letter in a school sport"]}, {"id": 576, "ent": "athlete", "stmt": "Athlete is a type of worker.", "wn_defs": ["a person trained to compete in sports"]}, {"id": 577, "ent": "hostage", "stmt": "Hostage is a type of game.", "wn_defs": ["a prisoner who is held by one party to insure that another party will meet specified terms"]}, {"id": 578, "ent": "empty", "stmt": "Empty is a type of container.", "wn_defs": ["a container that has been emptied"]}, {"id": 579, "ent": "spacecraft", "stmt": "Spacecraft is a type of machine.", "wn_defs": ["a craft capable of traveling in outer space; technically, a satellite around the sun"]}, {"id": 580, "ent": "kilt", "stmt": "Kilt is a type of boat.", "wn_defs": ["a knee-length pleated tartan skirt worn by men as part of the traditional dress in the Highlands of northern Scotland"]}, {"id": 581, "ent": "shook", "stmt": "Shook is a type of body part.", "wn_defs": ["a disassembled barrel; the parts packed for storage or shipment"]}, {"id": 582, "ent": "headscarf", "stmt": "Headscarf is a type of clothing.", "wn_defs": ["a kerchief worn over the head and tied under the chin"]}, {"id": 583, "ent": "geranium", "stmt": "Geranium is a type of animal.", "wn_defs": ["any of numerous plants of the family Geraniaceae"]}, {"id": 584, "ent": "mangrove", "stmt": "Mangrove is a type of weapon.", "wn_defs": ["a tropical tree or shrub bearing fruit that germinates while still on the tree and having numerous prop roots that eventually form an impenetrable mass and are important in land building"]}, {"id": 585, "ent": "librarian", "stmt": "Librarian is a type of food.", "wn_defs": ["a professional person trained in library science and engaged in library services"]}, {"id": 586, "ent": "restaurateur", "stmt": "Restaurateur is a type of athlete.", "wn_defs": ["the proprietor of a restaurant"]}, {"id": 587, "ent": "perfectionist", "stmt": "Perfectionist is a type of living thing.", "wn_defs": ["a person who is displeased by anything that does not meet very high standards"]}, {"id": 588, "ent": "flashlight", "stmt": "Flashlight is a type of tool.", "wn_defs": ["a small portable battery-powered electric lamp"]}, {"id": 589, "ent": "groundhog", "stmt": "Groundhog is a type of man-made object.", "wn_defs": ["reddish brown North American marmot"]}, {"id": 590, "ent": "contemplative", "stmt": "Contemplative is a type of footwear.", "wn_defs": ["a person devoted to the contemplative life"]}, {"id": 591, "ent": "robber", "stmt": "Robber is a type of living thing.", "wn_defs": ["a thief who steals from someone by threatening violence"]}, {"id": 592, "ent": "tequila", "stmt": "Tequila is a type of plant.", "wn_defs": ["Mexican liquor made from fermented juices of an agave plant"]}, {"id": 593, "ent": "expert", "stmt": "Expert is a type of person.", "wn_defs": ["a person with special knowledge or ability who performs skillfully"]}, {"id": 594, "ent": "wheelchair", "stmt": "Wheelchair is a type of game.", "wn_defs": ["a movable chair mounted on large wheels; for invalids or those who cannot walk; frequently propelled by the occupant"]}, {"id": 595, "ent": "cuttlefish", "stmt": "Cuttlefish is a type of artist.", "wn_defs": ["ten-armed oval-bodied cephalopod with narrow fins as long as the body and a large calcareous internal shell"]}, {"id": 596, "ent": "traveler", "stmt": "Traveler is a type of game.", "wn_defs": ["a person who changes location"]}, {"id": 597, "ent": "thermos", "stmt": "Thermos is a type of plant.", "wn_defs": ["vacuum flask that preserves temperature of hot or cold drinks"]}, {"id": 598, "ent": "customer", "stmt": "Customer is a type of weapon.", "wn_defs": ["someone who pays for goods or services"]}, {"id": 599, "ent": "loner", "stmt": "Loner is a type of living thing.", "wn_defs": ["a person who avoids the company or assistance of others"]}, {"id": 600, "ent": "slipknot", "stmt": "Slipknot is a type of material.", "wn_defs": ["a knot at the end of a cord or rope that can slip along the cord or rope around which it is made"]}, {"id": 601, "ent": "landlady", "stmt": "Landlady is a type of game.", "wn_defs": ["a landlord who is a woman"]}, {"id": 602, "ent": "oppressor", "stmt": "Oppressor is a type of device.", "wn_defs": ["a person of authority who subjects others to undue pressures"]}, {"id": 603, "ent": "alderman", "stmt": "Alderman is a type of boat.", "wn_defs": ["a member of a municipal legislative body (as a city council)"]}, {"id": 604, "ent": "fife", "stmt": "Fife is a type of fabric.", "wn_defs": ["a small high-pitched flute similar to a piccolo; has a shrill tone and is used chiefly to accompany drums in a marching band"]}, {"id": 605, "ent": "loudspeaker", "stmt": "Loudspeaker is a type of disease.", "wn_defs": ["electro-acoustic transducer that converts electrical signals into sounds loud enough to be heard at a distance"]}, {"id": 606, "ent": "revisionist", "stmt": "Revisionist is a type of artist.", "wn_defs": ["a Communist who tries to rewrite Marxism to justify a retreat from the revolutionary position"]}, {"id": 607, "ent": "plenipotentiary", "stmt": "Plenipotentiary is a type of mammal.", "wn_defs": ["a diplomat who is fully authorized to represent his or her government"]}, {"id": 608, "ent": "houseboat", "stmt": "Houseboat is a type of vehicle.", "wn_defs": ["a barge that is designed and equipped for use as a dwelling"]}, {"id": 609, "ent": "optimist", "stmt": "Optimist is a type of body part.", "wn_defs": ["a person disposed to take a favorable view of things"]}, {"id": 610, "ent": "primrose", "stmt": "Primrose is a type of flower.", "wn_defs": ["any of numerous short-stemmed plants of the genus Primula having tufted basal leaves and showy flowers clustered in umbels or heads"]}, {"id": 611, "ent": "moray", "stmt": "Moray is a type of insect.", "wn_defs": ["family of brightly colored voracious eels of warm coastal waters; generally nonaggressive to humans but larger species are dangerous if provoked"]}, {"id": 612, "ent": "roebuck", "stmt": "Roebuck is a type of animal.", "wn_defs": ["male roe deer"]}, {"id": 613, "ent": "cafe", "stmt": "Cafe is a type of man-made object.", "wn_defs": ["a small restaurant where drinks and snacks are sold"]}, {"id": 614, "ent": "proletarian", "stmt": "Proletarian is a type of worker.", "wn_defs": ["a member of the working class (not necessarily employed)"]}, {"id": 615, "ent": "limiter", "stmt": "Limiter is a type of man-made object.", "wn_defs": ["(electronics) a nonlinear electronic circuit whose output is limited in amplitude; used to limit the instantaneous amplitude of a waveform (to clip off the peaks of a waveform)"]}, {"id": 616, "ent": "brogan", "stmt": "Brogan is a type of man-made object.", "wn_defs": ["a thick and heavy shoe"]}, {"id": 617, "ent": "wingman", "stmt": "Wingman is a type of game.", "wn_defs": ["the pilot who positions his aircraft outside and behind (on the wing of) the leader of a flying formation"]}, {"id": 618, "ent": "robber", "stmt": "Robber is a type of game.", "wn_defs": ["a thief who steals from someone by threatening violence"]}, {"id": 619, "ent": "aquarium", "stmt": "Aquarium is a type of boat.", "wn_defs": ["a tank or pool or bowl filled with water for keeping live fish and underwater animals"]}, {"id": 620, "ent": "handkerchief", "stmt": "Handkerchief is a type of device.", "wn_defs": ["a square piece of cloth used for wiping the eyes or nose or as a costume accessory"]}, {"id": 621, "ent": "legislator", "stmt": "Legislator is a type of worker.", "wn_defs": ["someone who makes or enacts laws"]}, {"id": 622, "ent": "preteen", "stmt": "Preteen is a type of person.", "wn_defs": ["a preadolescent boy or girl (usually between 9 and 12 years of age)"]}, {"id": 623, "ent": "individualist", "stmt": "Individualist is a type of worker.", "wn_defs": ["a person who pursues independent thought or action"]}, {"id": 624, "ent": "detonator", "stmt": "Detonator is a type of furniture.", "wn_defs": ["a mechanical or electrical explosive device or a small amount of explosive; can be used to initiate the reaction of a disrupting explosive"]}, {"id": 625, "ent": "banjo", "stmt": "Banjo is a type of fish.", "wn_defs": ["a stringed instrument of the guitar family that has long neck and circular body"]}, {"id": 626, "ent": "neurosurgeon", "stmt": "Neurosurgeon is a type of footwear.", "wn_defs": ["someone who does surgery on the nervous system (especially the brain)"]}, {"id": 627, "ent": "sunflower", "stmt": "Sunflower is a type of living thing.", "wn_defs": ["any plant of the genus Helianthus having large flower heads with dark disk florets and showy yellow rays"]}, {"id": 628, "ent": "kylie", "stmt": "Kylie is a type of aircraft.", "wn_defs": ["an Australian boomerang; one side flat and the other convex"]}, {"id": 629, "ent": "nawab", "stmt": "Nawab is a type of clothing.", "wn_defs": ["a governor in India during the Mogul empire"]}, {"id": 630, "ent": "assistant", "stmt": "Assistant is a type of person.", "wn_defs": ["a person who contributes to the fulfillment of a need or furtherance of an effort or purpose"]}, {"id": 631, "ent": "cantilever", "stmt": "Cantilever is a type of body part.", "wn_defs": ["projecting horizontal beam fixed at one end only"]}, {"id": 632, "ent": "libertine", "stmt": "Libertine is a type of fish.", "wn_defs": ["a dissolute person; usually a man who is morally unrestrained"]}, {"id": 633, "ent": "chronicler", "stmt": "Chronicler is a type of artist.", "wn_defs": ["someone who writes chronicles"]}, {"id": 634, "ent": "earl", "stmt": "Earl is a type of boat.", "wn_defs": ["a British peer ranking below a marquess and above a viscount"]}, {"id": 635, "ent": "abbess", "stmt": "Abbess is a type of building.", "wn_defs": ["the superior of a group of nuns"]}, {"id": 636, "ent": "murine", "stmt": "Murine is a type of food.", "wn_defs": ["a rodent that is a member of the family Muridae"]}, {"id": 637, "ent": "stole", "stmt": "Stole is a type of weapon.", "wn_defs": ["a wide scarf worn about their shoulders by women"]}, {"id": 638, "ent": "stapler", "stmt": "Stapler is a type of boat.", "wn_defs": ["a machine that inserts staples into sheets of paper in order to fasten them together"]}, {"id": 639, "ent": "ski", "stmt": "Ski is a type of device.", "wn_defs": ["narrow wood or metal or plastic runners used in pairs for gliding over snow"]}, {"id": 640, "ent": "materiel", "stmt": "Materiel is a type of equipment.", "wn_defs": ["equipment and supplies of a military force"]}, {"id": 641, "ent": "rheumatic", "stmt": "Rheumatic is a type of living thing.", "wn_defs": ["a person suffering with rheumatism"]}, {"id": 642, "ent": "margarita", "stmt": "Margarita is a type of dish.", "wn_defs": ["a cocktail made of tequila and triple sec with lime and lemon juice"]}, {"id": 643, "ent": "dredge", "stmt": "Dredge is a type of animal.", "wn_defs": ["a power shovel to remove material from a channel or riverbed"]}, {"id": 644, "ent": "private", "stmt": "Private is a type of fruit.", "wn_defs": ["an enlisted man of the lowest rank in the Army or Marines"]}, {"id": 645, "ent": "chalet", "stmt": "Chalet is a type of car.", "wn_defs": ["a Swiss house with a sloping roof and wide eaves or a house built in this style"]}, {"id": 646, "ent": "locket", "stmt": "Locket is a type of device.", "wn_defs": ["a small ornamental case; usually contains a picture or a lock of hair and is worn on a necklace"]}, {"id": 647, "ent": "earl", "stmt": "Earl is a type of animal.", "wn_defs": ["a British peer ranking below a marquess and above a viscount"]}, {"id": 648, "ent": "blazer", "stmt": "Blazer is a type of weapon.", "wn_defs": ["lightweight single-breasted jacket; often striped in the colors of a club or school"]}, {"id": 649, "ent": "jammer", "stmt": "Jammer is a type of body part.", "wn_defs": ["a transmitter used to broadcast electronic jamming"]}, {"id": 650, "ent": "crustacean", "stmt": "Crustacean is a type of food.", "wn_defs": ["any mainly aquatic arthropod usually having a segmented body and chitinous exoskeleton"]}, {"id": 651, "ent": "android", "stmt": "Android is a type of worker.", "wn_defs": ["an automaton that resembles a human being"]}, {"id": 652, "ent": "cowgirl", "stmt": "Cowgirl is a type of animal.", "wn_defs": ["a woman cowboy"]}, {"id": 653, "ent": "entrepreneur", "stmt": "Entrepreneur is a type of food.", "wn_defs": ["someone who organizes a business venture and assumes the risk for it"]}, {"id": 654, "ent": "implant", "stmt": "Implant is a type of body part.", "wn_defs": ["a prosthesis placed permanently in tissue"]}, {"id": 655, "ent": "camcorder", "stmt": "Camcorder is a type of device.", "wn_defs": ["a portable television camera and videocassette recorder"]}, {"id": 656, "ent": "macadamia", "stmt": "Macadamia is a type of living thing.", "wn_defs": ["any tree of the genus Macadamia"]}, {"id": 657, "ent": "waterfowl", "stmt": "Waterfowl is a type of game.", "wn_defs": ["freshwater aquatic bird"]}, {"id": 658, "ent": "celibate", "stmt": "Celibate is a type of person.", "wn_defs": ["an unmarried person who has taken a religious vow of chastity"]}, {"id": 659, "ent": "rhinoceros", "stmt": "Rhinoceros is a type of person.", "wn_defs": ["massive powerful herbivorous odd-toed ungulate of southeast Asia and Africa having very thick skin and one or two horns on the snout"]}, {"id": 660, "ent": "blazer", "stmt": "Blazer is a type of man-made object.", "wn_defs": ["lightweight single-breasted jacket; often striped in the colors of a club or school"]}, {"id": 661, "ent": "retriever", "stmt": "Retriever is a type of game.", "wn_defs": ["a dog with heavy water-resistant coat that can be trained to retrieve game"]}, {"id": 662, "ent": "antelope", "stmt": "Antelope is a type of footwear.", "wn_defs": ["graceful Old World ruminant with long legs and horns directed upward and backward; includes gazelles; springboks; impalas; addax; gerenuks; blackbucks; dik-diks"]}, {"id": 663, "ent": "wally", "stmt": "Wally is a type of sport.", "wn_defs": ["a silly and inept person; someone who is regarded as stupid"]}, {"id": 664, "ent": "usurper", "stmt": "Usurper is a type of game.", "wn_defs": ["one who wrongfully or illegally seizes and holds the place of another"]}, {"id": 665, "ent": "cultivar", "stmt": "Cultivar is a type of material.", "wn_defs": ["a variety of a plant developed from a natural species and maintained under cultivation"]}, {"id": 666, "ent": "chalet", "stmt": "Chalet is a type of game.", "wn_defs": ["a Swiss house with a sloping roof and wide eaves or a house built in this style"]}, {"id": 667, "ent": "kitchen", "stmt": "Kitchen is a type of equipment.", "wn_defs": ["a room equipped for preparing meals"]}, {"id": 668, "ent": "matchbox", "stmt": "Matchbox is a type of toy.", "wn_defs": ["a box for holding matches"]}, {"id": 669, "ent": "homeowner", "stmt": "Homeowner is a type of person.", "wn_defs": ["someone who owns a home"]}, {"id": 670, "ent": "candlestick", "stmt": "Candlestick is a type of tool.", "wn_defs": ["a holder with sockets for candles"]}, {"id": 671, "ent": "priestess", "stmt": "Priestess is a type of dog.", "wn_defs": ["a woman priest"]}, {"id": 672, "ent": "toddler", "stmt": "Toddler is a type of mammal.", "wn_defs": ["a young child"]}, {"id": 673, "ent": "detonator", "stmt": "Detonator is a type of man-made object.", "wn_defs": ["a mechanical or electrical explosive device or a small amount of explosive; can be used to initiate the reaction of a disrupting explosive"]}, {"id": 674, "ent": "ax", "stmt": "Ax is a type of man-made object.", "wn_defs": ["an edge tool with a heavy bladed head mounted across a handle"]}, {"id": 675, "ent": "lingerie", "stmt": "Lingerie is a type of equipment.", "wn_defs": ["women's underwear and nightclothes"]}, {"id": 676, "ent": "biochemist", "stmt": "Biochemist is a type of worker.", "wn_defs": ["someone with special training in biochemistry"]}, {"id": 677, "ent": "clary", "stmt": "Clary is a type of weapon.", "wn_defs": ["aromatic herb of southern Europe; cultivated in Great Britain as a potherb and widely as an ornamental"]}, {"id": 678, "ent": "juvenile", "stmt": "Juvenile is a type of person.", "wn_defs": ["a young person, not fully developed"]}, {"id": 679, "ent": "psychologist", "stmt": "Psychologist is a type of worker.", "wn_defs": ["a scientist trained in psychology"]}, {"id": 680, "ent": "obstetrician", "stmt": "Obstetrician is a type of person.", "wn_defs": ["a physician specializing in obstetrics"]}, {"id": 681, "ent": "biologist", "stmt": "Biologist is a type of worker.", "wn_defs": ["(biology) a scientist who studies living organisms"]}, {"id": 682, "ent": "camcorder", "stmt": "Camcorder is a type of room.", "wn_defs": ["a portable television camera and videocassette recorder"]}, {"id": 683, "ent": "aristocrat", "stmt": "Aristocrat is a type of fruit.", "wn_defs": ["a member of the aristocracy"]}, {"id": 684, "ent": "tripod", "stmt": "Tripod is a type of equipment.", "wn_defs": ["a three-legged rack used for support"]}, {"id": 685, "ent": "ashtray", "stmt": "Ashtray is a type of metal.", "wn_defs": ["a receptacle for the ash from smokers' cigars or cigarettes"]}, {"id": 686, "ent": "atheist", "stmt": "Atheist is a type of dog.", "wn_defs": ["someone who denies the existence of god"]}, {"id": 687, "ent": "airship", "stmt": "Airship is a type of body part.", "wn_defs": ["a steerable self-propelled aircraft"]}, {"id": 688, "ent": "greenhouse", "stmt": "Greenhouse is a type of container.", "wn_defs": ["a building with glass walls and roof; for the cultivation and exhibition of plants under controlled conditions"]}, {"id": 689, "ent": "canoe", "stmt": "Canoe is a type of vehicle.", "wn_defs": ["small and light boat; pointed at both ends; propelled with a paddle"]}, {"id": 690, "ent": "ship", "stmt": "Ship is a type of tool.", "wn_defs": ["a vessel that carries passengers or freight"]}, {"id": 691, "ent": "jodhpur", "stmt": "Jodhpur is a type of vehicle.", "wn_defs": ["a short riding boot that fastens with a buckle at the side"]}, {"id": 692, "ent": "coffin", "stmt": "Coffin is a type of footwear.", "wn_defs": ["box in which a corpse is buried or cremated"]}, {"id": 693, "ent": "wrestler", "stmt": "Wrestler is a type of athlete.", "wn_defs": ["combatant who tries to throw opponent to the ground"]}, {"id": 694, "ent": "dominatrix", "stmt": "Dominatrix is a type of vehicle.", "wn_defs": ["a dominating woman (especially one who plays that role in a sadomasochistic sexual relationship)"]}, {"id": 695, "ent": "caribou", "stmt": "Caribou is a type of game.", "wn_defs": ["Arctic deer with large antlers in both sexes; called `reindeer' in Eurasia and `caribou' in North America"]}, {"id": 696, "ent": "harmonica", "stmt": "Harmonica is a type of equipment.", "wn_defs": ["a small rectangular free-reed instrument having a row of free reeds set back in air holes and played by blowing into the desired hole"]}, {"id": 697, "ent": "genitalia", "stmt": "Genitalia is a type of vegetable.", "wn_defs": ["external sex organ"]}, {"id": 698, "ent": "anesthesiologist", "stmt": "Anesthesiologist is a type of worker.", "wn_defs": ["a specialist who administers an anesthetic to a patient before he is treated"]}, {"id": 699, "ent": "golf", "stmt": "Golf is a type of equipment.", "wn_defs": ["a game played on a large open course with 9 or 18 holes; the object is use as few strokes as possible in playing all the holes"]}]''')
print(len(ITEMS), '個')

## 1. Wiktionaryの説明文を取る（5〜10分。取ったものはドライブに残し、もう一度実行すると取れていない語だけ取り直す）

In [ ]:
wk_path = f"{OUT}/E6_wiktionary.json"
words = sorted(set(it["ent"] for it in ITEMS))
import collections
WK = json.load(open(wk_path)) if os.path.exists(wk_path) else {}
todo = [w for w in words if not WK.get(w)]   # 取れていない語だけ取り直す（もう一度実行すると続きから）
print("取れている語:", len(words) - len(todo), " これから取る語:", len(todo))
S = requests.Session(); S.headers["User-Agent"] = "research-notebook/0.2 (non-commercial; python-requests)"
STATUS = collections.Counter(); FIRST_ERR = {}
def req(url, **kw):
    for k in range(6):                      # 断られたら待って取り直す
        try:
            r = S.get(url, timeout=30, **kw)
        except Exception as e:
            STATUS["通信エラー"] += 1; time.sleep(2 ** k); continue
        STATUS[r.status_code] += 1
        if r.status_code == 200:
            return r
        FIRST_ERR.setdefault(r.status_code, r.text[:300])
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(float(r.headers.get("Retry-After") or 0) or 2 ** (k + 1)); continue
        return None
    return None
def get_json(w):
    r = req("https://en.wiktionary.org/api/rest_v1/page/definition/" + urllib.parse.quote(w), params={"redirect": "true"})
    return r.json() if r is not None else None
def get_wikitext(w):  # 予備（上で取れなかった語だけ）
    r = req("https://en.wiktionary.org/w/api.php", params={"action": "parse", "page": w, "prop": "wikitext",
            "format": "json", "formatversion": 2, "redirects": 1})
    return r.json().get("parse", {}).get("wikitext", "") if r is not None else ""
for i in range(0, len(todo), 50):          # 50語ごとにドライブへ保存
    WK.update(fetch_all(todo[i:i + 50], get_json, get_wikitext, log=lambda s: None, sleep=0.5))
    json.dump(WK, open(wk_path, "w"), ensure_ascii=False)
    print(f"{min(i + 50, len(todo))}/{len(todo)}  取れた語の合計 {sum(1 for w in words if WK.get(w))}  返事の種類 {dict(STATUS)}")
for it in ITEMS: it["wk_defs"] = WK.get(it["ent"], [])
got = sum(1 for w in words if WK.get(w))
print("説明文が取れた語:", got, "/", len(words), " 返事の種類:", dict(STATUS))
print("例:", ITEMS[0]["ent"], WK.get(ITEMS[0]["ent"]))
for code_, txt in FIRST_ERR.items(): print("断られた例", code_, txt)
if got < len(words) * 0.7:
    raise RuntimeError("説明文の取れた語が7割未満です。この出力をClaudeに見せてください（もう一度このセルを実行すると続きから取ります）。")

## 2. Gemmaにもう一度聞く・NLIで判定する

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
def gemma_logits(qs, B=16):
    out = []
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        out += (last[:, YES] - last[:, NO]).cpu().numpy().tolist()
    return out
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORDER = [lab["entailment"], lab["neutral"], lab["contradiction"]]
def nli_probs(P, H, B=32):
    out = []
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORDER]
        out += pr.cpu().numpy().tolist()
    return out
t = time.time()
res = run(ITEMS, gemma_logits, nli_probs)
path = f"{OUT}/E6_results.json"
json.dump({"meta": {"mock": False, "gemma": GEMMA_ID, "nli": NLI_ID, "n": len(res)}, "items": res}, open(path, "w"), ensure_ascii=False)
print("保存:", path, f"{time.time() - t:.0f}秒")
g = np.array([[it["gemma"][k] for k in ["none", "S1", "S2", "S3"]] for it in res])
print("「はい」の割合 説明文なし/S1/S2/S3:", np.round((g > 0).mean(0), 3))